<a href="https://colab.research.google.com/github/TheGenesisAIStory/ml-trading-thesis-bot/blob/main/Company_Valuatio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. ⚙️ Setup & Config

In questa sezione configuriamo l'ambiente sperimentale per il motore DCF a 10 anni, definendo parametri, universo, logging e stile grafico coerente con una dashboard fintech moderna.

| Cell | What it does | Output |
|------|---------------|--------|
| 0.1  | Installa le dipendenze necessarie in modo silenzioso | Librerie pronte all'uso |
| 0.2  | Importa moduli, imposta il seed e lo stile grafico | Ambiente coerente e replicabile |
| 0.3  | Definisce EXPERIMENT, UNIVERSE e colori | Configurazione centrale dell'esperimento |
| 0.4  | Crea le cartelle di output e inizializza il logger | Struttura `output/` e log dell'esecuzione |

Il cuore economico di questo notebook è un modello di Discounted Cash Flow (DCF) multi‑periodo. Valutiamo ogni titolo come il valore attuale dei flussi di cassa futuri, scontati a un tasso coerente con il rischio:

\[
V_0 = \sum_{t=1}^{T} \frac{FCF_t}{(1 + r)^t} + \frac{TV}{(1 + r)^T}
\]

dove \(FCF_t\) è il free cash flow al tempo \(t\), \(r\) è il tasso di sconto (cost of equity o WACC) e \(TV\) è il terminal value, calcolato con modelli di crescita perpetua o multipli di mercato.

In [ ]:
# 0.1 Installazione dipendenze
import subprocess
import sys

def pip_install(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    except Exception as e:
        print(f"[WARNING] Failed to install {pkg}: {e}")

required_packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "python-dotenv",
    "tqdm",
    "scikit-learn",
    "plotly",
]

for pkg in required_packages:
    pip_install(pkg)

In [ ]:
# 0.2 Import, seed, stile grafici
import os
from pathlib import Path
import logging
import json
import importlib.util
import importlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

from tqdm import tqdm
from dotenv import load_dotenv


PLOTLY_AVAILABLE = importlib.util.find_spec("plotly") is not None
if PLOTLY_AVAILABLE:
    px = importlib.import_module("plotly.express")
    go = importlib.import_module("plotly.graph_objects")
else:
    px = None
    go = None

load_dotenv()
np.random.seed(42)

COLORS = {
    "primary":  "#01696f",
    "accent":   "#da7101",
    "q1":       "#c0392b",
    "neutral":  "#7a7974",
    "bg":       "#f7f6f2",
    "blue":     "#006494",
    "gold":     "#d19900",
    "purple":   "#7a39bb",
}

plt.rcParams["figure.dpi"] = 180
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["axes.facecolor"] = COLORS["bg"]
plt.rcParams["savefig.facecolor"] = COLORS["bg"]

In [102]:
# ============================================================
# 0.1 MASTER USER CONFIG
# notebook valuation / ML - enterprise grade
# ============================================================

USER_SELECTION = {
    # --------------------------------------------------------
    # A) DATA SOURCE / COMPANY
    # --------------------------------------------------------
    "data_source_choice": "europe_italy_market_data",
    # options:
    # "default"
    # "database_finanziario"
    # "ml_trading_repo"
    # "europe_stoxx_companies"
    # "europe_italy_market_data"
    # "sqlite_database"
    # "custom_folder"

    "selected_company": "ucg_mi",
    "custom_folder_path": None,

    # --------------------------------------------------------
    # B) USER EXPERIENCE
    # --------------------------------------------------------
    "user_profile": "cfa_balanced",
    # options:
    # "retail_simple"
    # "conservative"
    # "balanced"
    # "aggressive"
    # "cfa_balanced"
    # "cfa_advanced"

    # --------------------------------------------------------
    # C) MODEL SCOPE
    # --------------------------------------------------------
    "run_valuation_models": [
        "dcf",
        "relative_valuation",
        "residual_income",
        "dividend_discount_model",
        "economic_profit"
    ],
    # possible:
    # "dcf"
    # "relative_valuation"
    # "residual_income"
    # "dividend_discount_model"
    # "economic_profit"

    "run_ml_models": [
        "linear_regression",
        "ridge",
        "lasso",
        "elastic_net",
        "random_forest",
        "xgboost",
        "lightgbm",
        "catboost"
    ],
    # possible:
    # "linear_regression"
    # "ridge"
    # "lasso"
    # "elastic_net"
    # "random_forest"
    # "xgboost"
    # "lightgbm"
    # "catboost"

    "compare_models": True,
    "ensemble_models": False,

    # --------------------------------------------------------
    # D) TARGET DEFINITION
    # --------------------------------------------------------
    "valuation_target": "fair_value",
    # options:
    # "fair_value"
    # "expected_return_12m"
    # "upside_downside_pct"
    # "valuation_gap"

    "ml_task_type": "regression",
    # options:
    # "regression"
    # "classification"

    "model_selection_metric": "out_of_sample_rmse",
    "ranking_metric": "blended_score",

    # --------------------------------------------------------
    # E) PEERS
    # --------------------------------------------------------
    "peer_selection_method": "sector_country",
    # options:
    # "sector_only"
    # "sector_country"
    # "sector_size_country"
    # "manual"

    "n_peers": 7,
    "manual_peers": [],

    # --------------------------------------------------------
    # F) REPORTING
    # --------------------------------------------------------
    "save_intermediate_tables": True,
    "save_figures": True,
    "save_model_objects": False,
    "verbose_logging": True,
}

In [103]:
# ============================================================
# 0.2 PRESET MAP
# ============================================================

PRESET_MAP = {
    "retail_simple": {
        "discount_rate": 0.09,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.05,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 5,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.35,
        "sales_growth_rate": 0.04,
        "target_operating_margin": 0.15,
        "ml_primary_model": "random_forest",
        "n_peers": 5,
        "use_sector_default_assumptions": True,
    },
    "conservative": {
        "discount_rate": 0.10,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.055,
        "terminal_growth_rate": 0.015,
        "forecast_horizon_years": 5,
        "tax_rate": 0.30,
        "reinvestment_rate": 0.45,
        "sales_growth_rate": 0.03,
        "target_operating_margin": 0.13,
        "ml_primary_model": "elastic_net",
        "n_peers": 6,
        "use_sector_default_assumptions": True,
    },
    "balanced": {
        "discount_rate": 0.09,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.05,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 5,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.40,
        "sales_growth_rate": 0.05,
        "target_operating_margin": 0.16,
        "ml_primary_model": "random_forest",
        "n_peers": 5,
        "use_sector_default_assumptions": True,
    },
    "aggressive": {
        "discount_rate": 0.08,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.045,
        "terminal_growth_rate": 0.025,
        "forecast_horizon_years": 7,
        "tax_rate": 0.27,
        "reinvestment_rate": 0.50,
        "sales_growth_rate": 0.07,
        "target_operating_margin": 0.18,
        "ml_primary_model": "xgboost",
        "n_peers": 8,
        "use_sector_default_assumptions": True,
    },
    "cfa_balanced": {
        "discount_rate": 0.09,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.05,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 5,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.42,
        "sales_growth_rate": 0.05,
        "target_operating_margin": 0.17,
        "return_on_new_invested_capital": 0.11,
        "target_debt_to_capital": 0.30,
        "cost_of_debt": 0.045,
        "beta": 1.00,
        "ml_primary_model": "random_forest",
        "n_peers": 7,
        "use_sector_default_assumptions": True,
    },
    "cfa_advanced": {
        "discount_rate": 0.095,
        "risk_free_rate": 0.03,
        "market_risk_premium": 0.055,
        "terminal_growth_rate": 0.02,
        "forecast_horizon_years": 7,
        "tax_rate": 0.28,
        "reinvestment_rate": 0.45,
        "sales_growth_rate": 0.055,
        "target_operating_margin": 0.18,
        "return_on_new_invested_capital": 0.12,
        "target_debt_to_capital": 0.32,
        "cost_of_debt": 0.0475,
        "beta": 1.05,
        "ml_primary_model": "xgboost",
        "n_peers": 10,
        "use_sector_default_assumptions": True,
    },
}

In [104]:
# ============================================================
# 0.3 ADVANCED VALUATION / ML CONFIG
# ============================================================

VALUATION_CONFIG = {
    "dcf": {
        "cash_flow_definition": "fcff",
        # "fcff" / "fcfe"
        "discounting_approach": "wacc",
        # "wacc" / "cost_of_equity"
        "terminal_value_method": "gordon_growth",
        # "gordon_growth" / "exit_multiple"
        "use_midyear_convention": True,
        "normalize_margins": True,
        "normalize_working_capital": True,
        "normalize_capex": True,
    },
    "relative_valuation": {
        "multiple_set": ["pe", "ev_ebitda", "ev_sales", "pbv"],
        "aggregation_method": "median",
        # "mean" / "median" / "trimmed_mean"
        "outlier_filtering": True,
        "winsorize_percentile": 0.05,
    },
    "residual_income": {
        "book_value_anchor": True,
        "cost_of_equity_method": "capm",
    },
    "dividend_discount_model": {
        "enabled_only_for_dividend_payers": True,
        "dividend_growth_method": "historical_blended",
    },
    "economic_profit": {
        "capital_charge_method": "wacc_times_invested_capital",
    },
    "sum_of_the_parts": {
        "enabled_for_conglomerates_only": True,
        "segment_valuation_method": "blended",
    },
    "asset_based_valuation": {
        "use_tangible_book_focus": True,
        "apply_holding_discount": False,
    },
}

In [105]:
ML_CONFIG = {
    "feature_set_type": "fundamental_plus_market",
    # "fundamental_only"
    # "market_only"
    # "fundamental_plus_market"
    # "fundamental_market_macro"

    "feature_selection_method": "hybrid",
    # "none"
    # "correlation_filter"
    # "mutual_info"
    # "model_based"
    # "hybrid"

    "scaling_method": "robust",
    # "none"
    # "standard"
    # "minmax"
    # "robust"

    "train_window_months": 60,
    "validation_window_months": 12,
    "test_window_months": 12,
    "cross_validation_folds": 5,
    "time_series_split": True,
    "hyperparameter_tuning": True,
    "performance_metric_regression": "rmse",
    # "rmse" / "mae" / "mape" / "r2"

    "performance_metric_classification": "f1",
    # "accuracy" / "precision" / "recall" / "f1" / "roc_auc"

    "classification_labels": ["SELL", "HOLD", "BUY"],
    "prediction_horizon_months": 12,
}

In [128]:
# 0.3 Configurazione PROJECT_ROOT, EXPERIMENT, UNIVERSE

# Se lavori in Colab con Drive montato:
# PROJECT_ROOT = Path("/content/drive/MyDrive/machine-learning-for-trading").resolve()
# Se lavori in locale, puoi usare Path(".").resolve() dalla root del repo:
PROJECT_ROOT = Path(".").resolve()

import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

EXPERIMENT = {
    "name": "fundamental_valuation_dcf_definitive",
    "start_date": "2010-01-01",
    "end_date":   "2026-01-01",
    "target":     "future_excess_return_12m",
    "horizons":   [252],
    "test_start": "2018-01-01",
    "embargo":    21,
    "n_quantiles": 5,
    "cost_bps":   10.0,
    "models":     ["ols"],
    "run_ablation":  True,
    "run_backtest":  True,
    "save_figures":  True,
    "feature_blocks": [
        "dcf_inputs",
        "quality_ratios",
        "valuation_outputs"
    ],
    # selezione sorgente per il file pulito
    # "file_mode": "manual" oppure "manifest" se vuoi usare un manifest in futuro
    "file_mode": "manual",
    # placeholder: questi li imposterai tu in Section 1
    "clean_panel_path": None,
    # DCF specific
    "dcf_horizon_years": 10,
    "dcf_terminal_method": ["gordon_growth", "exit_multiple"],
    "dcf_perpetual_growth": 0.02,
    "dcf_exit_multiple_type": "ev_ebitda",
    "dcf_exit_multiple_default": 12.0,
    # Discount rate parameters
    "discount_rate_mode": "ke_capm",
    "risk_free_proxy": "US10Y",
    "equity_risk_premium": 0.05,
    "beta_lookback_years": 5,
    "active_universe": "best_universe_118",
    "price_end_date": "2026-05-13",
    "max_tickers_api": 200,
    "tax_rate": 0.25
}

UNIVERSES = {
    "best_universe_118": {
        "description": "118 titoli Best Universe (Finance Database)",
        "fund_file": "TimeSeriesCleanStocksItems.xlsx",
        "update_files": [
            "Aggiornamento database foundamentals Best Universe 2+.xlsx",
            "Aggiornamento database foundamentals Best Universe.xlsx"
        ],
        "price_source": "drive+yfinance"
    },
    "ftse_mib": {
        "description": "FTSE MIB 40 costituenti",
        "price_source": "yfinance",
        "suffix": ".MI"
    },
    "sp500": {
        "description": "S&P 500",
        "price_source": "yfinance",
        "suffix": ""
    },
    "eurostoxx50": {
        "description": "EuroStoxx 50",
        "fund_file": "EUROSTOXX50.xlsx",
        "price_source": "yfinance",
        "suffix": ""
    }
}

PROJECT_ROOT = /content


In [106]:
GOVERNANCE_CONFIG = {
    "random_seed": 42,
    "run_sensitivity_analysis": True,
    "sensitivity_discount_rate_bps": [-200, -100, 0, 100, 200],
    "sensitivity_terminal_growth_bps": [-100, 0, 100],
    "run_scenario_analysis": True,
    "scenario_names": ["bear", "base", "bull"],
    "generate_investment_view": True,
    "generate_model_comparison_table": True,
    "generate_peer_comparison_table": True,
    "generate_explainability_outputs": True,
    "compare_train_vs_test_metrics": True,
    "require_out_of_sample_validation": True,
    "store_model_ranking": True,
}

In [108]:
# ============================================================
# 0.6 RE-MERGE CONFIG AFTER MANUAL UPDATES
# ============================================================

selected_profile = USER_SELECTION["user_profile"]

if selected_profile not in PRESET_MAP:
    raise ValueError(
        f"user_profile '{selected_profile}' non valido. "
        f"Valori ammessi: {list(PRESET_MAP.keys())}"
    )

preset_values = PRESET_MAP[selected_profile]

EXPERIMENT.update(USER_SELECTION)
EXPERIMENT.update(preset_values)
EXPERIMENT["VALUATION_CONFIG"] = VALUATION_CONFIG
EXPERIMENT["ML_CONFIG"] = ML_CONFIG
EXPERIMENT["GOVERNANCE_CONFIG"] = GOVERNANCE_CONFIG

print("Configurazione aggiornata correttamente.")
print(f"Selected company: {EXPERIMENT['selected_company']}")
print(f"User profile: {EXPERIMENT['user_profile']}")
print(f"Valuation models: {EXPERIMENT['run_valuation_models']}")
print(f"ML models: {EXPERIMENT['run_ml_models']}")
print(f"Model selection metric: {EXPERIMENT.get('model_selection_metric', 'N/A')}")
print(f"Ranking metric: {EXPERIMENT.get('ranking_metric', 'N/A')}")

Configurazione aggiornata correttamente.
Selected company: ucg_mi
User profile: cfa_balanced
Valuation models: ['dcf', 'relative_valuation', 'residual_income', 'dividend_discount_model', 'economic_profit']
ML models: ['linear_regression', 'ridge', 'lasso', 'elastic_net', 'random_forest', 'xgboost', 'lightgbm', 'catboost']
Model selection metric: out_of_sample_rmse
Ranking metric: blended_score


In [ ]:
# ============================================================
# 0.6 bis - Reframe del notebook: company analysis + ML overlay
# ============================================================

EXPERIMENT["analysis_mode"] = "company_analysis_with_ml_overlay"
EXPERIMENT["primary_objective"] = "worth_investing_decision"
EXPERIMENT["target"] = "target_mispricing_score"
EXPERIMENT["secondary_target"] = "future_excess_return_12m"

ML_CONFIG["feature_set_type"] = "fundamental_plus_market"
ML_CONFIG["classification_labels"] = ["NOT_INVESTABLE", "INVESTABLE"]
ML_CONFIG["prediction_horizon_months"] = 12

EXPERIMENT["feature_blocks"] = [
    "dcf_inputs",
    "quality_ratios",
    "valuation_outputs",
    "market_overlay"
]

logger.info("[0.6 bis] Notebook riallineato a company_analysis_with_ml_overlay")
print("analysis_mode      :", EXPERIMENT["analysis_mode"])
print("primary_objective  :", EXPERIMENT["primary_objective"])
print("target             :", EXPERIMENT["target"])
print("secondary_target   :", EXPERIMENT["secondary_target"])
print("feature_set_type   :", ML_CONFIG["feature_set_type"])
print("class_labels       :", ML_CONFIG["classification_labels"])

In [109]:
# 0.4 Output folders e logging

OUTPUT_ROOT = PROJECT_ROOT / "output"
FIGURES_DIR = OUTPUT_ROOT / "figures"
TABLES_DIR  = OUTPUT_ROOT / "tables"
LOGS_DIR    = OUTPUT_ROOT / "logs"

for d in [OUTPUT_ROOT, FIGURES_DIR, TABLES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

log_file = LOGS_DIR / f"{EXPERIMENT['name']}.log"

logger = logging.getLogger(EXPERIMENT["name"])
logger.setLevel(logging.INFO)
logger.handlers = []

fh = logging.FileHandler(log_file)
fh.setLevel(logging.INFO)
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)

formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
fh.setFormatter(formatter)
ch.setFormatter(formatter)

logger.addHandler(fh)
logger.addHandler(ch)

logger.info("Experiment started")

2026-05-12 21:54:25,161 - INFO - Experiment started
INFO:fundamental_valuation_dcf_definitive:Experiment started


In [136]:
# Path canonici riusabili in tutto il notebook
OUTPUT_DIR = OUTPUT_ROOT
TABLES_OUTPUT_DIR = TABLES_DIR
FIGURES_OUTPUT_DIR = FIGURES_DIR
LOGS_OUTPUT_DIR = LOGS_DIR

## 1. 📥 Data Ingestion

In questa sezione carichiamo i dati finanziari direttamente dalla cartella **Database Finanziario** su Google Drive, usando gli ID file per un accesso robusto indipendente dal path di montaggio.

| Cell | What it does | Output |
|------|---------------|--------|
| 1.1  | Monta Drive e scarica i file del Database Finanziario via `gdown` | File locali in `/content/data_db/` |
| 1.2  | Loader intelligente: legge il file con più colonne (package > prices) | `df_panel` pronto per il DCF |
| 1.3  | Carica anche i risk factors per il calcolo del costo del capitale | `df_risk` |
| 1.4  | Data source summary | `Table_1_data_source_summary.csv` |

Il **Database Finanziario** è il data lake centralizzato del progetto: contiene prezzi giornalieri, fattori di rischio e dati fondamentali per l'universo azionario. Il pannello integrato permette di costruire un DCF completo per ogni titolo e periodo senza dover riconciliare manualmente sorgenti diverse.

$$
\text{Panel}_{i,t} = \{\,P_{i,t},\; \text{FCF}_{i,t},\; r^f_t,\; \beta_i,\; \text{ERP}_t\,\}
$$

In [110]:
from pathlib import Path

base = Path("/content/drive/MyDrive/Database Finanziario")

print("Contenuto di MyDrive (prime 30 voci):")
for p in list(base.glob("*"))[:30]:
    print(" -", p.name)

Contenuto di MyDrive (prime 30 voci):
 - Betting Against Beta Original Paper Data.xlsx
 - Betting Against Beta Equity Factors Daily.xlsx
 - Quality Minus Junk Factors Daily.xlsx
 - The Devil in HMLs Details Factors Daily.xlsx
 - Cripto_MinuteTimeSeries
 - europe_italy_market_data
 - europe_stoxx_companies
 - alternative_data
 - Finance Database
 - catalog
 - data_providers
 - alpha_factor_research
 - bayesian_machine_learning
 - deep_learning
 - convolutional_neural_nets
 - autoencoders
 - deep_reinforcement_learning
 - alpha_factor_library
 - data
 - ml_trading_risk_factors_csv
 - ml_trading_market_prices_csv
 - ml_trading_database_package.gsheet


In [111]:
# 1.1 Install dipendenze, monta Drive, scopre i file del Database Finanziario

import subprocess, sys, os, logging, shutil
from pathlib import Path
import numpy as np
import pandas as pd

for pkg in ["gdown", "yfinance", "pandas-datareader", "gspread",
            "google-auth", "openpyxl", "pyarrow", "tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Monta Drive
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    logger.info("Google Drive montato: /content/drive/MyDrive")
except Exception as e:
    logger.warning(f"Drive mount skipped: {e}")

# Trova DB_ROOT dinamicamente
mydrive = Path("/content/drive/MyDrive")
db_name = next(
    (n for n in os.listdir(str(mydrive))
     if "Database" in n and "Finanziario" in n), None
)
DB_ROOT = mydrive / db_name if db_name else None
logger.info(f"DB_ROOT: {DB_ROOT} | exists: {DB_ROOT.exists() if DB_ROOT else False}")

# Cartella locale di lavoro
DATA_LOCAL = Path("/content/data_db")
DATA_LOCAL.mkdir(parents=True, exist_ok=True)

# Repertorio file nel Database Finanziario
# ml_trading_market_prices_csv  → CSV con prezzi giornalieri
# ml_trading_risk_factors_csv   → CSV con fattori di rischio Fama-French
# ml_trading_database_package   → Google Sheet (accesso via gspread)
FILE_MAP = {
    "market_prices": {
        "drive_name": "ml_trading_market_prices_csv",
        "local"     : DATA_LOCAL / "ml_trading_market_prices.csv",
        "is_gsheet" : False,
        "required"  : True,
    },
    "risk_factors": {
        "drive_name": "ml_trading_risk_factors_csv",
        "local"     : DATA_LOCAL / "ml_trading_risk_factors.csv",
        "is_gsheet" : False,
        "required"  : False,
    },
    "db_package": {
        "drive_name": "ml_trading_database_package",
        "local"     : DATA_LOCAL / "ml_trading_database_package.csv",
        "is_gsheet" : True,
        "required"  : False,
    },
}

# Copia file CSV/Parquet da Drive
for key, cfg in FILE_MAP.items():
    if cfg["is_gsheet"]:
        continue  # gestito in 1.2
    local = cfg["local"]
    if local.exists():
        logger.info(f"[1.1] {key}: già in cache → {local}")
        continue
    if DB_ROOT:
        src = DB_ROOT / cfg["drive_name"]
        for ext in ["", ".csv", ".parquet", ".xlsx"]:
            candidate = Path(str(src) + ext)
            if candidate.exists() and not candidate.suffix == ".gsheet":
                try:
                    shutil.copy2(str(candidate), str(local))
                    logger.info(f"[1.1] {key}: copiato da {candidate.name}")
                    break
                except OSError as e:
                    logger.warning(f"[1.1] {key}: copia fallita ({e})")
        else:
            logger.warning(f"[1.1] {key}: non trovato in DB_ROOT")

EXPERIMENT["db_root"]  = str(DB_ROOT) if DB_ROOT else "N/A"
EXPERIMENT["file_map"] = {k: str(v["local"]) for k, v in FILE_MAP.items()}
EXPERIMENT["data_local"] = str(DATA_LOCAL)

print("=== FILE STATUS ===")
for key, cfg in FILE_MAP.items():
    p = cfg["local"]
    print(f"  {key:15s} | exists={str(p.exists()):5s} | {p.name}")

2026-05-12 21:55:00,986 - INFO - Google Drive montato: /content/drive/MyDrive
INFO:fundamental_valuation_dcf_definitive:Google Drive montato: /content/drive/MyDrive
2026-05-12 21:55:00,993 - INFO - DB_ROOT: /content/drive/MyDrive/Database Finanziario | exists: True
INFO:fundamental_valuation_dcf_definitive:DB_ROOT: /content/drive/MyDrive/Database Finanziario | exists: True
2026-05-12 21:55:00,996 - INFO - [1.1] market_prices: già in cache → /content/data_db/ml_trading_market_prices.csv
INFO:fundamental_valuation_dcf_definitive:[1.1] market_prices: già in cache → /content/data_db/ml_trading_market_prices.csv
2026-05-12 21:55:00,998 - INFO - [1.1] risk_factors: già in cache → /content/data_db/ml_trading_risk_factors.csv
INFO:fundamental_valuation_dcf_definitive:[1.1] risk_factors: già in cache → /content/data_db/ml_trading_risk_factors.csv


=== FILE STATUS ===
  market_prices   | exists=True  | ml_trading_market_prices.csv
  risk_factors    | exists=True  | ml_trading_risk_factors.csv
  db_package      | exists=False | ml_trading_database_package.csv


In [ ]:
# ============================================================
# 1.1 bis - Cache manager + persistence su Drive + manifest
# ============================================================

import os
import json
import shutil
import hashlib
from pathlib import Path
from datetime import datetime

DATA_LOCAL = Path("/content/data_db")
RAW_CACHE = DATA_LOCAL / "raw_cache"
PROCESSED_CACHE = DATA_LOCAL / "processed_cache"
REPORTS_CACHE = DATA_LOCAL / "reports"

for p in [DATA_LOCAL, RAW_CACHE, PROCESSED_CACHE, REPORTS_CACHE]:
    p.mkdir(parents=True, exist_ok=True)

DB_ROOT = Path(EXPERIMENT.get("db_root", "/content/drive/MyDrive/Database Finanziario"))
DB_ROOT = Path(DB_ROOT) if str(DB_ROOT) != "N/A" else None

if DB_ROOT and DB_ROOT.exists():
    DRIVE_RAW_CACHE = DB_ROOT / "raw_cache"
    DRIVE_PROCESSED_CACHE = DB_ROOT / "processed_cache"
    DRIVE_REPORTS = DB_ROOT / "reports"
    for p in [DRIVE_RAW_CACHE, DRIVE_PROCESSED_CACHE, DRIVE_REPORTS]:
        p.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_RAW_CACHE = None
    DRIVE_PROCESSED_CACHE = None
    DRIVE_REPORTS = None

MANIFEST_PATH_LOCAL = DATA_LOCAL / "run_manifest.json"
MANIFEST_PATH_DRIVE = (DB_ROOT / "run_manifest.json") if DB_ROOT and DB_ROOT.exists() else None

def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def persist_file(local_path, drive_dir=None, logger=None):
    local_path = Path(local_path)
    if logger is None:
        logger = logging.getLogger(__name__)

    meta = {
        "local_path": str(local_path),
        "exists": local_path.exists(),
        "size_bytes": None,
        "md5": None,
        "drive_path": None,
        "copied_to_drive": False,
        "updated_at": datetime.utcnow().isoformat()
    }

    if not local_path.exists():
        return meta

    meta["size_bytes"] = local_path.stat().st_size
    meta["md5"] = file_md5(local_path)

    if drive_dir is not None:
        drive_dir = Path(drive_dir)
        drive_dir.mkdir(parents=True, exist_ok=True)
        target = drive_dir / local_path.name
        try:
            shutil.copy2(local_path, target)
            meta["drive_path"] = str(target)
            meta["copied_to_drive"] = True
            logger.info(f"[persist] copiato {local_path.name} -> {target}")
        except Exception as e:
            logger.warning(f"[persist] copia fallita per {local_path.name}: {e}")

    return meta

RUN_MANIFEST = {
    "run_timestamp_utc": datetime.utcnow().isoformat(),
    "experiment_name": EXPERIMENT.get("name"),
    "db_root": str(DB_ROOT) if DB_ROOT else None,
    "datasets": {}
}

EXPERIMENT["data_local"] = str(DATA_LOCAL)
EXPERIMENT["raw_cache"] = str(RAW_CACHE)
EXPERIMENT["processed_cache"] = str(PROCESSED_CACHE)
EXPERIMENT["reports_cache"] = str(REPORTS_CACHE)

print("DATA_LOCAL:", DATA_LOCAL)
print("DB_ROOT   :", DB_ROOT)

In [113]:
# 1.2 Carica df_panel: Drive locale → API yfinance → sintetico

import yfinance as yf
from tqdm import tqdm

# Universe predefinito se non carichiamo da file
DEFAULT_TICKERS_SP500_SAMPLE = [
    "AAPL","MSFT","GOOGL","AMZN","META","NVDA","BRK-B","JPM","JNJ","V",
    "PG","UNH","HD","MA","DIS","PYPL","ADBE","NFLX","INTC","CSCO",
    "PFE","KO","PEP","ABT","TMO","NKE","MRK","WMT","BAC","XOM",
    "CVX","LLY","ABBV","AVGO","COST","DHR","ACN","TXN","NEE","UNP",
    "MDT","LIN","HON","PM","AMGN","BMY","UPS","SBUX","IBM","GE"
]

def load_prices_from_api(tickers, start, end, logger=None):
    if logger is None:
        logger = logging.getLogger(__name__)
    logger.info(f"[API] Scarico prezzi per {len(tickers)} ticker via yfinance")
    frames = []
    for t in tqdm(tickers, desc="yfinance download"):
        try:
            raw = yf.download(t, start=start, end=end,
                              auto_adjust=True, progress=False)
            if raw.empty:
                continue
            raw = raw[["Close","Volume"]].copy()
            raw.columns = ["adj_close","volume"]
            raw.index.name = "date"
            raw.reset_index(inplace=True)
            raw["ticker"] = t
            frames.append(raw)
        except Exception as e:
            logger.warning(f"[API] {t}: {e}")
    if not frames:
        raise RuntimeError("Nessun dato scaricato da yfinance")
    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values(["ticker","date"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    logger.info(f"[API] Panel scaricato: shape={df.shape}")
    return df

def load_dataframe_safe(path: Path, logger=None):
    if logger is None:
        logger = logging.getLogger(__name__)
    for candidate in [path,
                      Path(str(path)+".parquet"),
                      Path(str(path)+".csv")]:
        if not candidate.exists():
            continue
        try:
            if candidate.suffix in [".parquet",".pq"]:
                df = pd.read_parquet(candidate)
            elif candidate.suffix == ".csv":
                df = pd.read_csv(candidate, low_memory=False)
            elif candidate.suffix in [".xlsx",".xls"]:
                df = pd.read_excel(candidate)
            else:
                continue
            logger.info(f"Loaded {candidate.name} shape={df.shape}")
            return df, str(candidate)
        except Exception as e:
            logger.warning(f"Errore lettura {candidate}: {e}")
    return None, None

# ── Tentativo 1: file locale da DB ───────────────────────────────────────
df_panel = None
panel_source = None

prices_path = Path(EXPERIMENT["file_map"]["market_prices"])
df_panel, panel_path = load_dataframe_safe(prices_path, logger)

if df_panel is not None and df_panel.shape[1] >= 3:
    panel_source = "database_finanziario_local"
    logger.info(f"[1.2] Panel caricato da file locale: {panel_path}")
else:
    logger.warning("[1.2] File locale non valido o mancante → fallback API yfinance")
    # ── Tentativo 2: API yfinance ─────────────────────────────────────────
    try:
        df_panel = load_prices_from_api(
            tickers=DEFAULT_TICKERS_SP500_SAMPLE,
            start=EXPERIMENT["start_date"],
            end=EXPERIMENT["end_date"],
            logger=logger
        )
        panel_source = "yfinance_api"
        # Salva sul Database Finanziario per uso futuro
        save_path = DATA_LOCAL / "ml_trading_market_prices.csv"
        df_panel.to_csv(save_path, index=False)
        logger.info(f"[1.2] Panel scaricato da API e salvato in {save_path}")
        # Copia anche su Drive per aggiornamenti futuri
        if DB_ROOT and DB_ROOT.exists():
            drive_save = DB_ROOT / "ml_trading_market_prices_csv.csv"
            try:
                shutil.copy2(str(save_path), str(drive_save))
                logger.info(f"[1.2] Panel salvato su Drive: {drive_save}")
            except Exception as e:
                logger.warning(f"[1.2] Salvataggio su Drive fallito: {e}")
    except Exception as e:
        logger.error(f"[1.2] API yfinance fallita: {e}")
        # ── Fallback sintetico ────────────────────────────────────────────
        logger.warning("[1.2] Generazione panel SINTETICO")
        dates  = pd.date_range(EXPERIMENT["start_date"], EXPERIMENT["end_date"], freq="B")
        tickers= ["AAPL_synthetic","MSFT_synthetic","GOOGL_synthetic"]
        rows   = []
        for t in tickers:
            prices = 100 * np.exp(np.cumsum(np.random.normal(0.0003,0.015,len(dates))))
            for i,d in enumerate(dates):
                rows.append({"date":d,"ticker":t,
                             "adj_close":prices[i],"volume":1e6,
                             "adj_close_synthetic":True})
        df_panel = pd.DataFrame(rows)
        panel_source = "synthetic"
        logger.warning("[1.2] Panel sintetico creato")

# Normalizza colonne chiave
for old, new in [
    (next((c for c in df_panel.columns if c.lower() in
           ["datetime","timestamp"]), None), "date"),
    (next((c for c in df_panel.columns if c.lower() in
           ["symbol","asset","stock"]), None), "ticker"),
    (next((c for c in df_panel.columns if c.lower() in
           ["adj close","close","price"]), None), "adj_close"),
]:
    if old and old != new and old in df_panel.columns:
        df_panel.rename(columns={old: new}, inplace=True)

if "date" in df_panel.columns:
    df_panel["date"] = pd.to_datetime(df_panel["date"])
if all(c in df_panel.columns for c in ["ticker","date"]):
    df_panel.sort_values(["ticker","date"], inplace=True)
    df_panel.reset_index(drop=True, inplace=True)

panel_meta = {
    "source"   : panel_source,
    "synthetic": panel_source == "synthetic",
    "path"     : panel_path or "api/synthetic",
    "mode"     : "database_finanziario",
}

logger.info(f"[1.2] df_panel finale: shape={df_panel.shape} | source={panel_source}")
print(f"Shape  : {df_panel.shape}")
print(f"Source : {panel_source}")
print(f"Columns: {df_panel.columns.tolist()}")
print(df_panel.head())

2026-05-12 21:55:01,475 - INFO - Loaded ml_trading_market_prices.csv shape=(140823, 11)
INFO:fundamental_valuation_dcf_definitive:Loaded ml_trading_market_prices.csv shape=(140823, 11)
2026-05-12 21:55:01,478 - INFO - [1.2] Panel caricato da file locale: /content/data_db/ml_trading_market_prices.csv
INFO:fundamental_valuation_dcf_definitive:[1.2] Panel caricato da file locale: /content/data_db/ml_trading_market_prices.csv
2026-05-12 21:55:01,567 - INFO - [1.2] df_panel finale: shape=(140823, 11) | source=database_finanziario_local
INFO:fundamental_valuation_dcf_definitive:[1.2] df_panel finale: shape=(140823, 11) | source=database_finanziario_local


Shape  : (140823, 11)
Source : database_finanziario_local
Columns: ['date', 'ticker', 'open', 'high', 'low', 'adj_close', 'adjusted', 'volume', 'source', 'source_symbol', 'asset_class']
        date  ticker  open  high  low  adj_close  adjusted  volume  \
0 2007-01-02  A2A.MI   NaN   NaN  NaN        NaN  1.499469     NaN   
1 2007-01-03  A2A.MI   NaN   NaN  NaN        NaN  1.500938     NaN   
2 2007-01-04  A2A.MI   NaN   NaN  NaN        NaN  1.486251     NaN   
3 2007-01-05  A2A.MI   NaN   NaN  NaN        NaN  1.468628     NaN   
4 2007-01-08  A2A.MI   NaN   NaN  NaN        NaN  1.461285     NaN   

                             source source_symbol asset_class  
0  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
1  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
2  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
3  export_python:FTSEMIB_prices.csv        A2A.MI      equity  
4  export_python:FTSEMIB_prices.csv        A2A.MI      equity  


In [114]:
def choose_best_candidate(candidates):
    if not candidates:
        return None

    preferred_keywords = [
        "feature_engineering_factor_panel",
        "alpha_factor_panel",
        "equity_feature_panel",
        "boosting_model_panel",
        "linear_model_panel",
        "market_prices",
        "clean_panel",
    ]

    excluded_keywords = [
        "backtest",
        "weights",
        "unsupervised",
        "eigen_portfolio",
        "efficient_frontier",
        "constituents",
    ]

    scored = []

    for path in candidates:
        name = path.name.lower()
        score = 0

        # penalizza file non adatti
        for kw in excluded_keywords:
            if kw in name:
                score -= 100

        # premia file panel/factor adatti
        for i, kw in enumerate(preferred_keywords[::-1], start=1):
            if kw in name:
                score += i * 20

        # preferenza formati
        if path.suffix.lower() == ".parquet":
            score += 10
        elif path.suffix.lower() == ".csv":
            score += 5
        elif path.suffix.lower() in [".xlsx", ".xls"]:
            score += 2

        scored.append((score, path))

    scored = sorted(scored, key=lambda x: (-x[0], str(x[1])))
    return scored[0][1]

In [115]:
# ============================================================
# 1.2 bis - Universe builder da Yahoo: STOXX50 + SP500 + FTSEMIB
# ============================================================

import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

UNIVERSE_CONFIG = {
    "use_yahoo_universe_builder": True,
    "include_sp500": True,
    "include_stoxx50": True,
    "include_ftsemib": True,
    "max_download_threads": 8,
}

EXPERIMENT["universe_config"] = UNIVERSE_CONFIG


def normalize_yahoo_ticker(t):
    if pd.isna(t):
        return None
    t = str(t).strip().upper()
    t = t.replace("/", "-")
    return t


def get_sp500_constituents():
    urls = [
        "https://pkgstore.datahub.io/core/s-and-p-500-companies/constituents_csv/data/constituents_csv.csv",
        "https://datahub.io/core/s-and-p-500-companies/r/constituents.csv",
    ]

    last_err = None
    for url in urls:
        try:
            df = pd.read_csv(url)
            break
        except Exception as e:
            last_err = e
            df = None

    if df is None or df.empty:
        raise ValueError(f"Impossibile scaricare S&P 500 da DataHub: {last_err}")

    rename_map = {
        "Symbol": "ticker",
        "Name": "company_name",
        "Security": "company_name",
        "Sector": "sector",
        "GICS Sector": "sector",
        "GICS Sub-Industry": "industry",
        "Sub-Industry": "industry",
    }
    df = df.rename(columns=rename_map).copy()

    required = ["ticker"]
    for c in required:
        if c not in df.columns:
            raise ValueError("Colonna ticker non trovata nel dataset S&P 500")

    if "company_name" not in df.columns:
        df["company_name"] = None
    if "sector" not in df.columns:
        df["sector"] = None
    if "industry" not in df.columns:
        df["industry"] = None

    df["ticker"] = df["ticker"].astype(str).map(normalize_yahoo_ticker)
    df["index_name"] = "SP500"

    return df[["ticker", "company_name", "sector", "industry", "index_name"]]

def get_stoxx50_constituents():
    stoxx50 = [
        "ADS.DE","ADYEN.AS","AIR.PA","ALV.DE","ASML.AS","ABI.BR","BAS.DE","BAYN.DE",
        "BBVA.MC","BNP.PA","CS.PA","DAI.DE","DB1.DE","DG.PA","DHL.DE","ENEL.MI",
        "ENI.MI","IBE.MC","IFX.DE","INGA.AS","ISP.MI","KER.PA","LIN.DE","MC.PA",
        "MUV2.DE","NOKIA.HE","ORA.PA","PHIA.AS","SAF.PA","SAN.MC","SAN.PA","SAP.DE",
        "SCHN.SW","SIE.DE","STLAM.MI","SU.PA","TTE.PA","UCG.MI","ULVR.L","VOW3.DE",
        "AI.PA","ANX.MC","BAMI.MI","BMW.DE","CRH.L","DTE.DE","EL.PA","RMS.PA",
        "RI.PA","PRX.AS"
    ]

    df = pd.DataFrame({"ticker": stoxx50})
    df["ticker"] = df["ticker"].map(normalize_yahoo_ticker)
    df["company_name"] = None
    df["sector"] = None
    df["industry"] = None
    df["index_name"] = "EURO_STOXX_50"
    return df[["ticker", "company_name", "sector", "industry", "index_name"]]

def get_ftsemib_constituents():
    manual_ftsemib = [
        "A2A.MI","AMP.MI","AZM.MI","BMPS.MI","BAMI.MI","BGN.MI","BMED.MI","BREB.MI",
        "BZU.MI","CPR.MI","DIA.MI","ENEL.MI","ENI.MI","FBK.MI","FER.MI","G.MI",
        "HER.MI","IG.MI","INW.MI","IP.MI","ISP.MI","IVG.MI","LDO.MI","MB.MI",
        "MONC.MI","MPS.MI","NEXI.MI","PIRC.MI","PST.MI","PRY.MI","RACE.MI","REC.MI",
        "SFER.MI","SRG.MI","STLAM.MI","STMMI.MI","TEN.MI","TIT.MI","TRN.MI","UCG.MI"
    ]
    df = pd.DataFrame({"ticker": manual_ftsemib})
    df["ticker"] = df["ticker"].map(normalize_yahoo_ticker)
    df["company_name"] = None
    df["sector"] = None
    df["industry"] = None
    df["index_name"] = "FTSEMIB"
    return df[["ticker", "company_name", "sector", "industry", "index_name"]]


universe_parts = []

for loader_name, loader_fn in [
    ("SP500", get_sp500_constituents),
    ("EURO STOXX 50", get_stoxx50_constituents),
    ("FTSEMIB", get_ftsemib_constituents),
]:
    try:
        df_idx = loader_fn()
        universe_parts.append(df_idx)
        logger.info(f"[Universe] {loader_name} constituents caricati: {len(df_idx)}")
    except Exception as e:
        logger.warning(f"[Universe] {loader_name} non disponibile: {e}")

if not universe_parts:
    raise RuntimeError("Nessun universo disponibile")

df_universe = pd.concat(universe_parts, ignore_index=True)
df_universe = df_universe.dropna(subset=["ticker"]).copy()
df_universe["ticker"] = df_universe["ticker"].map(normalize_yahoo_ticker)
df_universe = df_universe.drop_duplicates(subset=["ticker"]).reset_index(drop=True)

EXPERIMENT["universe_size"] = int(len(df_universe))
EXPERIMENT["universe_indices"] = sorted(df_universe["index_name"].dropna().unique().tolist())

print("Universe size:", len(df_universe))
print("Indices:", EXPERIMENT["universe_indices"])
display(df_universe.groupby("index_name")["ticker"].count().reset_index(name="n_tickers"))
display(df_universe.head())

2026-05-12 21:55:05,037 - INFO - [Universe] SP500 constituents caricati: 503
INFO:fundamental_valuation_dcf_definitive:[Universe] SP500 constituents caricati: 503
2026-05-12 21:55:05,042 - INFO - [Universe] EURO STOXX 50 constituents caricati: 50
INFO:fundamental_valuation_dcf_definitive:[Universe] EURO STOXX 50 constituents caricati: 50
2026-05-12 21:55:05,048 - INFO - [Universe] FTSEMIB constituents caricati: 40
INFO:fundamental_valuation_dcf_definitive:[Universe] FTSEMIB constituents caricati: 40


Universe size: 587
Indices: ['EURO_STOXX_50', 'FTSEMIB', 'SP500']


,index_name,n_tickers
0,EURO_STOXX_50,50
1,FTSEMIB,34
2,SP500,503


,ticker,company_name,sector,industry,index_name
0,MMM,3M,Industrials,Industrial Conglomerates,SP500
1,AOS,A. O. Smith,Industrials,Building Products,SP500
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,SP500
3,ABBV,AbbVie,Health Care,Biotechnology,SP500
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,SP500


In [ ]:
# ============================================================
# 1.2 ter - Persist universe constituents
# ============================================================

universe_local_csv = PROCESSED_CACHE / "universe_constituents_master.csv"
universe_local_parquet = PROCESSED_CACHE / "universe_constituents_master.parquet"
universe_by_index_csv = PROCESSED_CACHE / "universe_constituents_by_index_counts.csv"

df_universe.to_csv(universe_local_csv, index=False)
df_universe.to_parquet(universe_local_parquet, index=False)

df_universe_counts = (
    df_universe.groupby("index_name")["ticker"]
    .nunique()
    .reset_index(name="n_tickers")
)
df_universe_counts.to_csv(universe_by_index_csv, index=False)

RUN_MANIFEST["datasets"]["universe_constituents_master_csv"] = persist_file(
    universe_local_csv, DRIVE_PROCESSED_CACHE, logger
)
RUN_MANIFEST["datasets"]["universe_constituents_master_parquet"] = persist_file(
    universe_local_parquet, DRIVE_PROCESSED_CACHE, logger
)
RUN_MANIFEST["datasets"]["universe_constituents_by_index_counts_csv"] = persist_file(
    universe_by_index_csv, DRIVE_REPORTS, logger
)

print("Universe constituents salvati in cache locale e Drive.")
display(df_universe_counts)

In [ ]:
# ============================================================
# 1.3 - FMP secret loading and fundamentals runtime config
# ============================================================

from pathlib import Path
import os
import json
import time
import requests
from tqdm import tqdm

FUNDAMENTALS_CONFIG = {
    "mode": "local_if_exists_else_api",
    "period": "annual",
    "limit": 20,
    "request_pause_seconds": 0.20,
    "fmp_base_url": "https://financialmodelingprep.com/api/v3",
    "local_csv_path": str(DATA_LOCAL / "fundamentals_api_panel.csv"),
    "local_parquet_path": str(DATA_LOCAL / "fundamentals_api_panel.parquet"),
    "drive_csv_path": str(DB_ROOT / "fundamentals_api_panel.csv") if "DB_ROOT" in globals() and DB_ROOT and DB_ROOT.exists() else None,
    "fundamental_reporting_lag_days": 90,
    "additional_conservative_lag_days": 1,
    "minimum_match_coverage_warning": 0.05,
}
EXPERIMENT["fundamentals_config"] = FUNDAMENTALS_CONFIG
EXPERIMENT["fundamentals_mode"] = FUNDAMENTALS_CONFIG["mode"]


def get_fmp_api_key(secret_names=("FMP_API_KEY", "fmp_api_key"), logger=None):
    """Load FMP API key from Colab secrets first, then environment variables.

    The function is intentionally quiet for missing individual Colab secret names:
    a warning is emitted only if no supported source is available.
    """
    if logger is None:
        logger = logging.getLogger(__name__)

    # Colab secrets are optional outside Colab.
    try:
        from google.colab import userdata
        for name in secret_names:
            try:
                value = userdata.get(name)
            except Exception:
                value = None
            if value and str(value).strip():
                return str(value).strip(), name
    except Exception:
        pass

    for name in secret_names:
        value = os.getenv(name)
        if value and value.strip():
            return value.strip(), f"env:{name}"

    logger.warning("FMP API key unavailable. Supported names: FMP_API_KEY, fmp_api_key. Fundamentals API download will be skipped if no local cache exists.")
    return None, None


FMP_API_KEY, FMP_SECRET_SOURCE = get_fmp_api_key(logger=logger)
FMP_API_KEY_AVAILABLE = bool(FMP_API_KEY)

EXPERIMENT["fmp_api_key_available"] = FMP_API_KEY_AVAILABLE
EXPERIMENT["fmp_secret_source"] = FMP_SECRET_SOURCE

local_fundamental_paths = [
    Path(FUNDAMENTALS_CONFIG["local_parquet_path"]),
    Path(FUNDAMENTALS_CONFIG["local_csv_path"]),
]
if FUNDAMENTALS_CONFIG["drive_csv_path"]:
    local_fundamental_paths.append(Path(FUNDAMENTALS_CONFIG["drive_csv_path"]))

print("fundamentals_mode:", FUNDAMENTALS_CONFIG["mode"])
print("local fundamentals exists:", any(p.exists() for p in local_fundamental_paths))
print("FMP_API_KEY available:", FMP_API_KEY_AVAILABLE)
print("secret source:", FMP_SECRET_SOURCE)


In [ ]:
# ============================================================
# 1.4 - Build df_fund from local cache or FMP statements
# ============================================================

import numpy as np
import pandas as pd


def normalize_ticker(value):
    if pd.isna(value):
        return None
    return str(value).strip().upper().replace("/", "-")


def get_panel_tickers(df_panel, max_tickers=None):
    if "df_panel" not in globals() or df_panel is None or df_panel.empty:
        raise ValueError("df_panel is required before fundamentals ingestion.")
    ticker_col = next((c for c in df_panel.columns if str(c).lower() in ["ticker", "symbol", "asset", "stock"]), None)
    if ticker_col is None:
        raise ValueError(f"No ticker column found in df_panel. Available columns: {df_panel.columns.tolist()}")
    tickers = pd.Series(df_panel[ticker_col].dropna().map(normalize_ticker).unique()).dropna().tolist()
    tickers = sorted(set(tickers))
    if max_tickers:
        tickers = tickers[: int(max_tickers)]
    return tickers


def read_fundamentals_cache(paths):
    for path in paths:
        path = Path(path)
        if not path.exists():
            continue
        try:
            if path.suffix.lower() in [".parquet", ".pq"]:
                out = pd.read_parquet(path)
            else:
                out = pd.read_csv(path, low_memory=False)
            if out is not None and not out.empty and "ticker" in out.columns:
                logger.info("[1.4] Loaded fundamentals cache %s shape=%s", path, out.shape)
                return out, str(path)
        except Exception as exc:
            logger.warning("[1.4] Could not read fundamentals cache %s: %s", path, exc)
    return pd.DataFrame(), None


def fmp_symbol_candidates(ticker):
    ticker = normalize_ticker(ticker)
    candidates = [ticker]
    if "." in ticker:
        base, suffix = ticker.split(".", 1)
        candidates.extend([ticker.replace(".", "-"), base])
        # FMP commonly uses exchange suffixes, but this keeps fallback defensive for vendors/caches.
        if suffix == "MI":
            candidates.extend([f"{base}.MIL", f"{base}.MI"])
    return list(dict.fromkeys([c for c in candidates if c]))


def fetch_fmp_statement(symbol, statement_endpoint, api_key, period="annual", limit=20):
    url = f"{FUNDAMENTALS_CONFIG['fmp_base_url']}/{statement_endpoint}/{symbol}"
    params = {"period": period, "limit": int(limit), "apikey": api_key}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise ValueError(payload.get("Error Message"))
    if not isinstance(payload, list):
        return pd.DataFrame()
    return pd.DataFrame(payload)


def fetch_fmp_fundamentals_for_ticker(ticker, api_key):
    endpoints = {
        "income": "income-statement",
        "balance": "balance-sheet-statement",
        "cashflow": "cash-flow-statement",
    }
    last_error = None
    for candidate in fmp_symbol_candidates(ticker):
        statement_frames = {}
        try:
            for statement_name, endpoint in endpoints.items():
                frame = fetch_fmp_statement(
                    candidate,
                    endpoint,
                    api_key=api_key,
                    period=FUNDAMENTALS_CONFIG["period"],
                    limit=FUNDAMENTALS_CONFIG["limit"],
                )
                time.sleep(float(FUNDAMENTALS_CONFIG["request_pause_seconds"]))
                if frame.empty:
                    statement_frames[statement_name] = pd.DataFrame()
                    continue
                frame = frame.copy()
                frame["statement_type"] = statement_name
                statement_frames[statement_name] = frame
        except Exception as exc:
            last_error = f"{candidate}: {exc}"
            continue

        if all(frame.empty for frame in statement_frames.values()):
            last_error = f"{candidate}: empty API returns"
            continue

        merged = None
        merge_keys = ["date", "symbol", "calendarYear", "period"]
        for statement_name, frame in statement_frames.items():
            if frame.empty:
                continue
            keep_keys = [k for k in merge_keys if k in frame.columns]
            value_cols = [c for c in frame.columns if c not in keep_keys + ["link", "finalLink"]]
            frame = frame[keep_keys + value_cols].copy()
            rename = {c: f"{statement_name}_{c}" for c in value_cols if c not in ["acceptedDate", "fillingDate"]}
            frame = frame.rename(columns=rename)
            if merged is None:
                merged = frame
            else:
                shared = [k for k in keep_keys if k in merged.columns and k in frame.columns]
                if not shared:
                    shared = ["date"] if "date" in merged.columns and "date" in frame.columns else None
                merged = merged.merge(frame, on=shared, how="outer", suffixes=("", f"_{statement_name}"))

        if merged is None or merged.empty:
            last_error = f"{candidate}: no mergeable statements"
            continue
        merged["ticker"] = normalize_ticker(ticker)
        merged["fmp_symbol_used"] = candidate
        merged["fundamentals_source"] = "fmp_api"
        return merged, {"ticker": ticker, "status": "success", "fmp_symbol_used": candidate, "rows": int(len(merged)), "error": None}

    return pd.DataFrame(), {"ticker": ticker, "status": "failed", "fmp_symbol_used": None, "rows": 0, "error": last_error}


def coalesce_columns(df, candidates, output_col):
    existing = [c for c in candidates if c in df.columns]
    if not existing:
        df[output_col] = np.nan
        return df
    df[output_col] = df[existing].bfill(axis=1).iloc[:, 0]
    return df


def standardize_fundamentals(raw):
    if raw is None or raw.empty:
        return pd.DataFrame()
    df = raw.copy()
    df.columns = [str(c).strip() for c in df.columns]
    ticker_col = next((c for c in df.columns if c.lower() in ["ticker", "symbol"]), None)
    if ticker_col is None:
        raise ValueError(f"Fundamentals missing ticker/symbol column. Columns: {df.columns.tolist()[:30]}")
    df["ticker"] = df[ticker_col].map(normalize_ticker)

    date_candidates = ["date", "income_date", "statement_date", "fiscalDateEnding", "calendarDate"]
    filing_candidates = ["acceptedDate", "income_acceptedDate", "balance_acceptedDate", "cashflow_acceptedDate", "fillingDate", "income_fillingDate", "balance_fillingDate", "cashflow_fillingDate", "filingDate"]
    accepted_candidates = ["acceptedDate", "income_acceptedDate", "balance_acceptedDate", "cashflow_acceptedDate"]

    df = coalesce_columns(df, date_candidates, "statement_date")
    df = coalesce_columns(df, filing_candidates, "fundamental_available_date")
    df = coalesce_columns(df, accepted_candidates, "accepted_date")

    df["statement_date"] = pd.to_datetime(df["statement_date"], errors="coerce").dt.tz_localize(None)
    df["fundamental_available_date"] = pd.to_datetime(df["fundamental_available_date"], errors="coerce").dt.tz_localize(None)
    df["accepted_date"] = pd.to_datetime(df["accepted_date"], errors="coerce").dt.tz_localize(None)

    fallback_available = df["statement_date"] + pd.to_timedelta(int(FUNDAMENTALS_CONFIG["fundamental_reporting_lag_days"]), unit="D")
    df["fundamental_available_date"] = df["accepted_date"].combine_first(df["fundamental_available_date"]).combine_first(fallback_available)
    df["effective_fundamental_date"] = df["fundamental_available_date"] + pd.to_timedelta(int(FUNDAMENTALS_CONFIG["additional_conservative_lag_days"]), unit="D")

    rename_candidates = {
        "revenue": ["income_revenue", "revenue", "totalRevenue"],
        "gross_profit": ["income_grossProfit", "grossProfit", "gross_profit"],
        "ebitda": ["income_ebitda", "ebitda"],
        "ebit": ["income_ebit", "ebit", "operatingIncome"],
        "net_income": ["income_netIncome", "netIncome", "net_income"],
        "eps": ["income_eps", "eps"],
        "total_assets": ["balance_totalAssets", "totalAssets", "total_assets"],
        "total_liabilities": ["balance_totalLiabilities", "totalLiabilities", "total_liabilities"],
        "total_equity": ["balance_totalStockholdersEquity", "totalStockholdersEquity", "total_equity", "bookValue"],
        "cash_and_equivalents": ["balance_cashAndCashEquivalents", "cashAndCashEquivalents", "cash", "totalCash"],
        "total_debt": ["balance_totalDebt", "totalDebt", "total_debt"],
        "operating_cash_flow": ["cashflow_operatingCashFlow", "operatingCashFlow", "operating_cash_flow"],
        "capital_expenditure": ["cashflow_capitalExpenditure", "capitalExpenditure", "capital_expenditure", "capex"],
        "free_cash_flow": ["cashflow_freeCashFlow", "freeCashFlow", "free_cash_flow"],
        "shares_outstanding": ["income_weightedAverageShsOut", "weightedAverageShsOut", "sharesOutstanding", "shares_outstanding"],
    }
    for canonical, candidates in rename_candidates.items():
        df = coalesce_columns(df, candidates, canonical)
        df[canonical] = pd.to_numeric(df[canonical], errors="coerce")

    df["book_value_per_share"] = df["total_equity"] / df["shares_outstanding"].replace(0, np.nan)
    df["debt_to_equity"] = df["total_debt"] / df["total_equity"].replace(0, np.nan)
    df["roe"] = df["net_income"] / df["total_equity"].replace(0, np.nan)
    df["roa"] = df["net_income"] / df["total_assets"].replace(0, np.nan)
    df["gross_margin"] = df["gross_profit"] / df["revenue"].replace(0, np.nan)
    df["net_margin"] = df["net_income"] / df["revenue"].replace(0, np.nan)
    df["revenue_growth"] = df.sort_values(["ticker", "statement_date"]).groupby("ticker")["revenue"].pct_change()

    provenance_cols = [c for c in ["fmp_symbol_used", "fundamentals_source", "calendarYear", "period"] if c in df.columns]
    canonical_cols = [
        "ticker", "statement_date", "fundamental_available_date", "accepted_date", "effective_fundamental_date",
        "revenue", "gross_profit", "ebitda", "ebit", "net_income", "eps", "total_assets", "total_liabilities",
        "total_equity", "cash_and_equivalents", "total_debt", "operating_cash_flow", "capital_expenditure",
        "free_cash_flow", "shares_outstanding", "book_value_per_share", "debt_to_equity", "roe", "roa",
        "gross_margin", "net_margin", "revenue_growth",
    ]
    keep_cols = list(dict.fromkeys(canonical_cols + provenance_cols))
    df = df[[c for c in keep_cols if c in df.columns]].copy()
    df = df.dropna(subset=["ticker", "effective_fundamental_date"])
    df = df.sort_values(["ticker", "effective_fundamental_date", "statement_date"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["ticker", "effective_fundamental_date", "statement_date"], keep="last")
    return df


selected_tickers = get_panel_tickers(df_panel, max_tickers=EXPERIMENT.get("max_tickers_api"))
logger.info("[1.4] Fundamentals universe from df_panel: %d tickers", len(selected_tickers))

cache_paths = [Path(FUNDAMENTALS_CONFIG["local_parquet_path"]), Path(FUNDAMENTALS_CONFIG["local_csv_path"])]
if FUNDAMENTALS_CONFIG["drive_csv_path"]:
    cache_paths.append(Path(FUNDAMENTALS_CONFIG["drive_csv_path"]))

raw_fund, raw_fund_source = read_fundamentals_cache(cache_paths)
fundamental_fetch_log = []

if raw_fund.empty and FMP_API_KEY_AVAILABLE:
    frames = []
    for ticker in tqdm(selected_tickers, desc="FMP fundamentals"):
        frame, status = fetch_fmp_fundamentals_for_ticker(ticker, FMP_API_KEY)
        fundamental_fetch_log.append(status)
        if not frame.empty:
            frames.append(frame)
    raw_fund = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    raw_fund_source = "fmp_api" if not raw_fund.empty else None
elif raw_fund.empty:
    logger.warning("[1.4] No fundamentals cache and no FMP API key. Notebook will continue in market-only mode.")

if not raw_fund.empty:
    df_fund = standardize_fundamentals(raw_fund)
else:
    df_fund = pd.DataFrame()

dffund = df_fund.copy()
failed_fundamental_tickers = pd.DataFrame(fundamental_fetch_log)
if not failed_fundamental_tickers.empty:
    failed_fundamental_tickers = failed_fundamental_tickers[failed_fundamental_tickers["status"].ne("success")].copy()

EXPERIMENT["fundamentals_available"] = bool(not df_fund.empty)
EXPERIMENT["fundamentals_source"] = raw_fund_source
EXPERIMENT["fundamentals_failed_tickers"] = failed_fundamental_tickers["ticker"].tolist() if not failed_fundamental_tickers.empty else []

if not df_fund.empty:
    local_csv = Path(FUNDAMENTALS_CONFIG["local_csv_path"])
    local_parquet = Path(FUNDAMENTALS_CONFIG["local_parquet_path"])
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    df_fund.to_csv(local_csv, index=False)
    try:
        df_fund.to_parquet(local_parquet, index=False)
    except Exception as exc:
        logger.warning("[1.4] Parquet save skipped: %s", exc)
    if FUNDAMENTALS_CONFIG["drive_csv_path"]:
        try:
            df_fund.to_csv(FUNDAMENTALS_CONFIG["drive_csv_path"], index=False)
        except Exception as exc:
            logger.warning("[1.4] Drive CSV save skipped: %s", exc)

fetch_log_path = TABLES_DIR / "fundamentals_fetch_log.csv"
pd.DataFrame(fundamental_fetch_log).to_csv(fetch_log_path, index=False)

print("fundamentals_available:", EXPERIMENT["fundamentals_available"])
print("fundamentals_source:", EXPERIMENT["fundamentals_source"])
print("df_fund shape:", df_fund.shape)
print("failed tickers:", len(EXPERIMENT["fundamentals_failed_tickers"]))
if not df_fund.empty:
    display(df_fund.head())
else:
    print("No fundamentals were assembled; downstream sections will use market-only features.")


## 2. 🧹 Cleaning / Integration

This section standardizes the market panel and performs a time-aware market/fundamentals merge. Fundamental observations are aligned by `effective_fundamental_date`, defined as the earliest available filing timestamp (`accepted_date` preferred, then filing date, then statement date plus a conservative reporting lag) plus an additional configurable lag. The as-of merge only uses rows with `effective_fundamental_date <= market date`, preventing look-ahead bias.

In [ ]:
# ============================================================
# 2.1 - Standardize market panel
# ============================================================

def require_columns(df, required, name):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns {missing}. Available: {df.columns.tolist()}")


def standardize_market_panel(raw_panel):
    if raw_panel is None or raw_panel.empty:
        raise ValueError("df_panel is empty; cannot continue.")
    df = raw_panel.copy()
    df.columns = [str(c).strip() for c in df.columns]
    date_col = next((c for c in df.columns if c.lower() in ["date", "datetime", "timestamp"]), None)
    ticker_col = next((c for c in df.columns if c.lower() in ["ticker", "symbol", "asset", "stock"]), None)
    if date_col is None or ticker_col is None:
        raise ValueError(f"df_panel requires date and ticker columns. Available columns: {df.columns.tolist()}")
    df = df.rename(columns={date_col: "date", ticker_col: "ticker"})
    price_candidates = ["adj_close", "adj close", "adjusted", "close", "price"]
    price_col = next((c for c in df.columns if c.lower() in price_candidates and pd.to_numeric(df[c], errors="coerce").notna().any()), None)
    if price_col is None:
        numeric_cols = [c for c in df.select_dtypes(include="number").columns if c.lower() not in ["volume", "id"]]
        if not numeric_cols:
            raise ValueError("No usable price column found in df_panel.")
        price_col = numeric_cols[0]
        logger.warning("[2.1] Falling back to numeric price column: %s", price_col)
    if price_col != "adj_close":
        df["adj_close"] = df[price_col]
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.tz_localize(None)
    df["ticker"] = df["ticker"].map(normalize_ticker)
    df["adj_close"] = pd.to_numeric(df["adj_close"], errors="coerce")
    if "volume" in df.columns:
        df["volume"] = pd.to_numeric(df["volume"], errors="coerce")
    df = df.dropna(subset=["date", "ticker", "adj_close"]).copy()
    df = df.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last").reset_index(drop=True)
    if df.empty:
        raise ValueError("df_panel has no valid rows after standardization.")
    return df, price_col


df, selected_price_col = standardize_market_panel(df_panel)
df_market = df.copy()

logger.info("[2.1] df_market shape=%s price_col=%s", df_market.shape, selected_price_col)
print("df_market shape:", df_market.shape)
print("date range:", df_market["date"].min().date(), "→", df_market["date"].max().date())
print("tickers:", df_market["ticker"].nunique())
display(df_market[["date", "ticker", "adj_close"]].head())


In [ ]:
# ============================================================
# 2.2 - Time-aware market + fundamentals merge (no look-ahead)
# ============================================================

MERGE_CONFIG = {
    "use_fundamentals": bool(EXPERIMENT.get("fundamentals_available", False)) and "df_fund" in globals() and df_fund is not None and not df_fund.empty,
    "staleness_warning_days": 550,
}
EXPERIMENT["merge_config"] = MERGE_CONFIG


def prepare_fundamentals_for_merge(df_fund):
    fund = standardize_fundamentals(df_fund)
    require_columns(fund, ["ticker", "effective_fundamental_date"], "df_fund")
    fund = fund.dropna(subset=["ticker", "effective_fundamental_date"]).copy()
    fund = fund.sort_values(["ticker", "effective_fundamental_date", "statement_date"])
    fund = fund.drop_duplicates(subset=["ticker", "effective_fundamental_date"], keep="last")
    return fund.reset_index(drop=True)


def grouped_asof_market_fundamentals(market, fund):
    merged_parts = []
    fundamental_cols = [c for c in fund.columns if c != "ticker"]
    for ticker, left in market.groupby("ticker", sort=False):
        right = fund.loc[fund["ticker"].eq(ticker)].copy()
        left = left.sort_values("date").copy()
        if right.empty:
            for col in fundamental_cols:
                if col not in left.columns:
                    left[col] = pd.NaT if "date" in col else np.nan
            merged_parts.append(left)
            continue
        right = right.sort_values("effective_fundamental_date")
        merged = pd.merge_asof(
            left,
            right,
            left_on="date",
            right_on="effective_fundamental_date",
            direction="backward",
            suffixes=("", "_fund"),
        )
        if "ticker_fund" in merged.columns:
            merged = merged.drop(columns=["ticker_fund"])
        merged_parts.append(merged)
    return pd.concat(merged_parts, ignore_index=True).sort_values(["ticker", "date"]).reset_index(drop=True)


if MERGE_CONFIG["use_fundamentals"]:
    df_fund_merge = prepare_fundamentals_for_merge(df_fund)
    df_merged = grouped_asof_market_fundamentals(df_market, df_fund_merge)
    df_merged["fundamental_matched"] = df_merged["effective_fundamental_date"].notna()
    df_merged["days_since_fundamental"] = (df_merged["date"] - df_merged["effective_fundamental_date"]).dt.days
    lookahead_rows = df_merged.loc[df_merged["days_since_fundamental"].lt(0)]
    if not lookahead_rows.empty:
        raise AssertionError("Look-ahead detected: fundamentals matched after market date.")
    coverage = df_merged["fundamental_matched"].mean()
    EXPERIMENT["merge_mode"] = "market_plus_fundamentals"
    EXPERIMENT["fundamental_merge_coverage"] = float(coverage)
    logger.info("[2.2] merge coverage %.2f%%", 100 * coverage)
    if coverage < float(FUNDAMENTALS_CONFIG["minimum_match_coverage_warning"]):
        logger.warning("[2.2] Low fundamentals merge coverage: %.2f%%", 100 * coverage)
else:
    df_fund_merge = pd.DataFrame()
    df_merged = df_market.copy()
    df_merged["fundamental_matched"] = False
    df_merged["days_since_fundamental"] = np.nan
    EXPERIMENT["merge_mode"] = "market_only"
    EXPERIMENT["fundamental_merge_coverage"] = 0.0
    logger.warning("[2.2] Fundamentals unavailable: using market-only dataset.")

# Compatibility aliases used by later sections.
df_dcf_ready = df_merged.copy()
df_dcf_merged = df_merged.copy()
df_dcf_filtered = df_merged.copy()

print("merge_mode:", EXPERIMENT["merge_mode"])
print("df_merged shape:", df_merged.shape)
print("fundamental coverage:", f"{EXPERIMENT['fundamental_merge_coverage']:.2%}")
if df_merged["fundamental_matched"].any():
    display(df_merged["days_since_fundamental"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).to_frame("days_since_fundamental"))
display(df_merged.head())


In [ ]:
# ============================================================
# 2.3 - Integration quality checks
# ============================================================

qa_rows = []

def add_qa(check_name, status, details):
    qa_rows.append({"check_name": check_name, "status": status, "details": details})

add_qa("market_non_empty", "PASS" if not df_market.empty else "FAIL", f"rows={len(df_market)}")
add_qa("market_unique_ticker_date", "PASS" if not df_market.duplicated(["ticker", "date"]).any() else "FAIL", f"duplicates={int(df_market.duplicated(['ticker', 'date']).sum())}")
add_qa("fundamentals_available", "PASS" if EXPERIMENT.get("fundamentals_available") else "WARN", f"rows={len(df_fund) if 'df_fund' in globals() and df_fund is not None else 0}")
add_qa("merge_no_lookahead", "PASS" if not df_merged["days_since_fundamental"].dropna().lt(0).any() else "FAIL", "effective_fundamental_date <= date for all matched rows")
add_qa("merge_coverage", "PASS" if EXPERIMENT["fundamental_merge_coverage"] >= FUNDAMENTALS_CONFIG["minimum_match_coverage_warning"] or EXPERIMENT["merge_mode"] == "market_only" else "WARN", f"coverage={EXPERIMENT['fundamental_merge_coverage']:.2%}")
if df_merged["fundamental_matched"].any():
    stale_pct = df_merged["days_since_fundamental"].gt(MERGE_CONFIG["staleness_warning_days"]).mean()
    add_qa("fundamental_staleness", "PASS" if stale_pct < 0.25 else "WARN", f"pct_gt_{MERGE_CONFIG['staleness_warning_days']}d={stale_pct:.2%}")

qa_df = pd.DataFrame(qa_rows)
qa_path = TABLES_DIR / "qa_notebook_checks.csv"
qa_df.to_csv(qa_path, index=False)
display(qa_df)


## 3. 🔧 Features

In [ ]:
# ============================================================
# 3.1 - Market and fundamental feature engineering
# ============================================================

feature_df = df_merged.copy().sort_values(["ticker", "date"]).reset_index(drop=True)

for window in [1, 5, 21, 63, 126, 252]:
    feature_df[f"ret_{window}d"] = feature_df.groupby("ticker")["adj_close"].pct_change(window).shift(1)

for window in [21, 63, 126]:
    one_day_ret = feature_df.groupby("ticker")["adj_close"].pct_change()
    feature_df[f"vol_{window}d"] = one_day_ret.groupby(feature_df["ticker"]).transform(lambda s: s.rolling(window).std() * np.sqrt(252)).shift(1)

feature_df["mom_12_1"] = feature_df.groupby("ticker")["adj_close"].pct_change(252).shift(21)
feature_df["price_sma50"] = feature_df["adj_close"] / feature_df.groupby("ticker")["adj_close"].transform(lambda s: s.rolling(50).mean()).shift(1)
feature_df["price_sma200"] = feature_df["adj_close"] / feature_df.groupby("ticker")["adj_close"].transform(lambda s: s.rolling(200).mean()).shift(1)

if "volume" in feature_df.columns:
    feature_df["volume"] = pd.to_numeric(feature_df["volume"], errors="coerce")
    feature_df["vol_ratio_21d"] = feature_df["volume"] / feature_df.groupby("ticker")["volume"].transform(lambda s: s.rolling(21).mean()).shift(1)

if EXPERIMENT["merge_mode"] == "market_plus_fundamentals":
    if "shares_outstanding" in feature_df.columns:
        feature_df["market_cap_est"] = feature_df["adj_close"] * feature_df["shares_outstanding"]
    if "eps" in feature_df.columns:
        feature_df["pe_ratio"] = feature_df["adj_close"] / feature_df["eps"].replace(0, np.nan)
    if "book_value_per_share" in feature_df.columns:
        feature_df["pb_ratio"] = feature_df["adj_close"] / feature_df["book_value_per_share"].replace(0, np.nan)
    if all(c in feature_df.columns for c in ["market_cap_est", "total_debt", "cash_and_equivalents", "ebitda"]):
        enterprise_value = feature_df["market_cap_est"] + feature_df["total_debt"].fillna(0) - feature_df["cash_and_equivalents"].fillna(0)
        feature_df["ev_ebitda"] = enterprise_value / feature_df["ebitda"].replace(0, np.nan)

rank_features = [c for c in ["ret_21d", "ret_126d", "mom_12_1", "vol_63d", "pe_ratio", "pb_ratio", "ev_ebitda", "roe", "roa", "debt_to_equity", "gross_margin", "net_margin", "revenue_growth"] if c in feature_df.columns]
for col in rank_features:
    feature_df[f"{col}_xrank"] = feature_df.groupby("date")[col].rank(pct=True)

# Winsorize model features only, preserving raw identifiers and prices.
protected_cols = {"date", "ticker", "adj_close", "volume", "statement_date", "fundamental_available_date", "accepted_date", "effective_fundamental_date"}
numeric_feature_cols = [c for c in feature_df.select_dtypes(include="number").columns if c not in protected_cols]
for col in numeric_feature_cols:
    lo, hi = feature_df[col].quantile([0.01, 0.99])
    if pd.notna(lo) and pd.notna(hi) and lo < hi:
        feature_df[col] = feature_df[col].clip(lo, hi)

df_feat = feature_df.copy()
dffeat = df_feat.copy()
feat_path = TABLES_DIR / "df_feat.parquet"
try:
    df_feat.to_parquet(feat_path, index=False)
except Exception as exc:
    logger.warning("[3.1] Could not save parquet feature set: %s", exc)
    df_feat.to_csv(TABLES_DIR / "df_feat.csv", index=False)

missingness = (df_feat[numeric_feature_cols].isna().mean().sort_values(ascending=False).reset_index())
missingness.columns = ["feature", "pct_missing"]
missingness.to_csv(TABLES_DIR / "Table_III_missingness.csv", index=False)

print("df_feat shape:", df_feat.shape)
print("numeric features:", len(numeric_feature_cols))
display(missingness.head(15))


## 4. 🎯 Targets

In [ ]:
# ============================================================
# 4.1 - Forward return targets and labels
# ============================================================

df_model = df_feat.copy().sort_values(["ticker", "date"]).reset_index(drop=True)
target_horizons = sorted(set(EXPERIMENT.get("horizons", [252]) + [1, 5, 21, 63, 126, 252]))

for horizon in target_horizons:
    future_price = df_model.groupby("ticker")["adj_close"].shift(-horizon)
    df_model[f"fwdret_{horizon}d"] = future_price / df_model["adj_close"] - 1.0

TARGET_HORIZON = int(EXPERIMENT.get("horizons", [252])[0])
target_col = f"fwdret_{TARGET_HORIZON}d"
EXPERIMENT["target_horizon_days"] = TARGET_HORIZON
EXPERIMENT["target_column"] = target_col

df_model["target_regression"] = df_model[target_col]
df_model["target_binary"] = df_model.groupby("date")[target_col].transform(lambda s: (s > s.median()).astype(float) if s.notna().sum() >= 2 else np.nan)

def safe_quantile_label(s, q=5):
    valid = s.dropna()
    if len(valid) < q:
        return pd.Series(np.nan, index=s.index)
    try:
        return pd.qcut(s.rank(method="first"), q=q, labels=False, duplicates="drop").astype(float) + 1
    except Exception:
        return pd.Series(np.nan, index=s.index)

df_model["target_quantile"] = df_model.groupby("date")[target_col].transform(safe_quantile_label)
df_model.to_csv(TABLES_DIR / "df_model_targets.csv", index=False)

print("target_col:", target_col)
print("df_model shape:", df_model.shape)
display(df_model[["date", "ticker", "adj_close", target_col, "target_binary", "target_quantile"]].head())


## 5. 📊 Descriptive Statistics

This section converts the integrated panel into an analyst-ready cross-section and computes the key availability, valuation, quality, momentum, and risk diagnostics used by the rest of the notebook.

In [ ]:
# ============================================================
# 5.1 - Analyst-ready cross-section, SWS-style scorecard, and dataset summary
# ============================================================

logger.info("[5.1] Building analyst cross-section, SWS-style checks, and descriptive tables")

def pct_fmt(x):
    return "n/a" if pd.isna(x) else f"{x:.2%}"


def safe_rank(series, ascending=True):
    valid = pd.to_numeric(series, errors="coerce")
    if valid.notna().sum() == 0:
        return pd.Series(np.nan, index=series.index)
    return valid.rank(pct=True, ascending=ascending)


def latest_per_ticker(frame):
    if frame.empty:
        return frame.copy()
    return frame.sort_values(["ticker", "date"]).groupby("ticker", as_index=False).tail(1).reset_index(drop=True)


def numeric_median(frame, col):
    if col not in frame.columns:
        return np.nan
    values = pd.to_numeric(frame[col], errors="coerce")
    return values.replace([np.inf, -np.inf], np.nan).median(skipna=True)


def binary_check(condition, index):
    return pd.Series(condition, index=index).astype(float).where(pd.Series(condition, index=index).notna(), np.nan)


def add_sws_style_scores(frame):
    """Create Simply Wall St-inspired 0-6 axis scores using available notebook fields.

    The public SWS framework groups checks into Value, Future Performance, Past Performance,
    Health, and Dividends/Income. This notebook implements transparent proxies for these axes
    so the logic remains reproducible and GitHub-ready even when analyst estimates/dividends are absent.
    """
    out = frame.copy()
    if "debt_to_assets" not in out.columns and all(c in out.columns for c in ["total_debt", "total_assets"]):
        out["debt_to_assets"] = out["total_debt"] / out["total_assets"].replace(0, np.nan)
    if "fcf_margin" not in out.columns and all(c in out.columns for c in ["free_cash_flow", "revenue"]):
        out["fcf_margin"] = out["free_cash_flow"] / out["revenue"].replace(0, np.nan)
    if "cash_to_debt" not in out.columns and all(c in out.columns for c in ["cash_and_equivalents", "total_debt"]):
        out["cash_to_debt"] = out["cash_and_equivalents"] / out["total_debt"].replace(0, np.nan)

    pe_median = numeric_median(out, "pe_ratio")
    pb_median = numeric_median(out, "pb_ratio")
    ev_median = numeric_median(out, "ev_ebitda")
    roe_median = numeric_median(out, "roe")
    roa_median = numeric_median(out, "roa")
    revenue_growth_median = numeric_median(out, "revenue_growth")
    debt_equity_median = numeric_median(out, "debt_to_equity")
    debt_assets_median = numeric_median(out, "debt_to_assets")
    gross_margin_median = numeric_median(out, "gross_margin")

    idx = out.index
    value_checks, future_checks, past_checks, health_checks, income_checks = [], [], [], [], []

    if "pe_ratio" in out.columns and pd.notna(pe_median):
        value_checks.append(binary_check((out["pe_ratio"] > 0) & (out["pe_ratio"] < pe_median), idx).rename("value_pe_below_universe"))
    if "pb_ratio" in out.columns and pd.notna(pb_median):
        value_checks.append(binary_check((out["pb_ratio"] > 0) & (out["pb_ratio"] < pb_median), idx).rename("value_pb_below_universe"))
    if "ev_ebitda" in out.columns and pd.notna(ev_median):
        value_checks.append(binary_check((out["ev_ebitda"] > 0) & (out["ev_ebitda"] < ev_median), idx).rename("value_ev_ebitda_below_universe"))
    if all(c in out.columns for c in ["pe_ratio", "revenue_growth"]):
        out["peg_proxy"] = out["pe_ratio"] / (out["revenue_growth"].replace(0, np.nan) * 100)
        value_checks.append(binary_check((out["peg_proxy"] > 0) & (out["peg_proxy"] <= 1.5), idx).rename("value_peg_proxy_reasonable"))

    if "revenue_growth" in out.columns:
        future_checks.append(binary_check(out["revenue_growth"] > 0, idx).rename("future_revenue_growth_positive"))
        if pd.notna(revenue_growth_median):
            future_checks.append(binary_check(out["revenue_growth"] > revenue_growth_median, idx).rename("future_revenue_growth_above_universe"))
    if "net_margin" in out.columns:
        future_checks.append(binary_check(out["net_margin"] > 0, idx).rename("future_net_margin_positive"))
    if "ret_126d" in out.columns:
        future_checks.append(binary_check(out["ret_126d"] > 0, idx).rename("future_price_trend_positive"))
    if "mom_12_1" in out.columns:
        future_checks.append(binary_check(out["mom_12_1"] > 0, idx).rename("future_long_momentum_positive"))

    if "roe" in out.columns:
        past_checks.append(binary_check(out["roe"] > 0, idx).rename("past_roe_positive"))
        past_checks.append(binary_check(out["roe"] >= 0.20, idx).rename("past_roe_above_20pct"))
        if pd.notna(roe_median):
            past_checks.append(binary_check(out["roe"] > roe_median, idx).rename("past_roe_above_universe"))
    if "roa" in out.columns:
        past_checks.append(binary_check(out["roa"] > 0, idx).rename("past_roa_positive"))
        if pd.notna(roa_median):
            past_checks.append(binary_check(out["roa"] > roa_median, idx).rename("past_roa_above_universe"))
    if "ret_252d" in out.columns:
        past_checks.append(binary_check(out["ret_252d"] > 0, idx).rename("past_12m_return_positive"))

    if "debt_to_equity" in out.columns:
        health_checks.append(binary_check(out["debt_to_equity"] < 1, idx).rename("health_debt_to_equity_below_1x"))
        if pd.notna(debt_equity_median):
            health_checks.append(binary_check(out["debt_to_equity"] < debt_equity_median, idx).rename("health_debt_to_equity_below_universe"))
    if "debt_to_assets" in out.columns and pd.notna(debt_assets_median):
        health_checks.append(binary_check(out["debt_to_assets"] < debt_assets_median, idx).rename("health_debt_to_assets_below_universe"))
    if "total_equity" in out.columns:
        health_checks.append(binary_check(out["total_equity"] > 0, idx).rename("health_positive_equity"))
    if "cash_to_debt" in out.columns:
        health_checks.append(binary_check(out["cash_to_debt"] > 0.25, idx).rename("health_cash_covers_debt_buffer"))
    if "days_since_fundamental" in out.columns:
        health_checks.append(binary_check(out["days_since_fundamental"].fillna(np.inf) <= MERGE_CONFIG.get("staleness_warning_days", 550), idx).rename("health_fundamentals_not_stale"))

    if "free_cash_flow" in out.columns:
        income_checks.append(binary_check(out["free_cash_flow"] > 0, idx).rename("income_fcf_positive"))
    if "operating_cash_flow" in out.columns:
        income_checks.append(binary_check(out["operating_cash_flow"] > 0, idx).rename("income_operating_cf_positive"))
    if "fcf_margin" in out.columns:
        income_checks.append(binary_check(out["fcf_margin"] > 0, idx).rename("income_fcf_margin_positive"))
    if "gross_margin" in out.columns and pd.notna(gross_margin_median):
        income_checks.append(binary_check(out["gross_margin"] > gross_margin_median, idx).rename("income_gross_margin_above_universe"))
    if all(c in out.columns for c in ["free_cash_flow", "net_income"]):
        out["fcf_to_net_income"] = out["free_cash_flow"] / out["net_income"].replace(0, np.nan)
        income_checks.append(binary_check(out["fcf_to_net_income"] > 0, idx).rename("income_fcf_to_net_income_positive"))

    check_groups = {
        "sws_value_score": value_checks,
        "sws_future_score": future_checks,
        "sws_past_score": past_checks,
        "sws_health_score": health_checks,
        "sws_income_score": income_checks,
    }
    for axis, checks in check_groups.items():
        if checks:
            check_df = pd.concat(checks[:6], axis=1)
            out[axis] = check_df.sum(axis=1, min_count=1).clip(0, 6)
            out[f"{axis}_available_checks"] = check_df.notna().sum(axis=1)
        else:
            out[axis] = np.nan
            out[f"{axis}_available_checks"] = 0
    axis_cols = list(check_groups.keys())
    out["sws_snowflake_score"] = out[axis_cols].mean(axis=1, skipna=True)
    out["sws_snowflake_score_pct"] = out["sws_snowflake_score"] / 6.0
    return out

latest_cross_section = latest_per_ticker(df_model)
if latest_cross_section.empty:
    raise ValueError("latest_cross_section is empty; earlier data assembly failed.")

valuation_inputs = []
for col in ["pe_ratio", "pb_ratio", "ev_ebitda"]:
    if col in latest_cross_section.columns:
        latest_cross_section[f"{col}_value_rank"] = safe_rank(latest_cross_section[col], ascending=True)
        valuation_inputs.append(f"{col}_value_rank")

quality_inputs = []
for col in ["roe", "roa", "gross_margin", "net_margin"]:
    if col in latest_cross_section.columns:
        latest_cross_section[f"{col}_quality_rank"] = safe_rank(latest_cross_section[col], ascending=False)
        quality_inputs.append(f"{col}_quality_rank")
if "debt_to_equity" in latest_cross_section.columns:
    latest_cross_section["debt_to_equity_quality_rank"] = safe_rank(latest_cross_section["debt_to_equity"], ascending=True)
    quality_inputs.append("debt_to_equity_quality_rank")

momentum_inputs = []
for col in ["ret_126d", "ret_252d", "mom_12_1"]:
    if col in latest_cross_section.columns:
        latest_cross_section[f"{col}_momentum_rank"] = safe_rank(latest_cross_section[col], ascending=False)
        momentum_inputs.append(f"{col}_momentum_rank")

risk_inputs = []
for col in ["vol_63d", "vol_126d"]:
    if col in latest_cross_section.columns:
        latest_cross_section[f"{col}_risk_rank"] = safe_rank(latest_cross_section[col], ascending=True)
        risk_inputs.append(f"{col}_risk_rank")

score_blocks = {}
if valuation_inputs:
    latest_cross_section["valuation_score"] = latest_cross_section[valuation_inputs].mean(axis=1)
    score_blocks["valuation_score"] = 0.30
if quality_inputs:
    latest_cross_section["quality_score"] = latest_cross_section[quality_inputs].mean(axis=1)
    score_blocks["quality_score"] = 0.20
if momentum_inputs:
    latest_cross_section["momentum_score"] = latest_cross_section[momentum_inputs].mean(axis=1)
    score_blocks["momentum_score"] = 0.20
if risk_inputs:
    latest_cross_section["risk_score"] = latest_cross_section[risk_inputs].mean(axis=1)
    score_blocks["risk_score"] = 0.10

latest_cross_section = add_sws_style_scores(latest_cross_section)
if latest_cross_section["sws_snowflake_score_pct"].notna().any():
    score_blocks["sws_snowflake_score_pct"] = 0.20

if score_blocks:
    weight_sum = sum(score_blocks.values())
    latest_cross_section["blended_score"] = sum(latest_cross_section[col].fillna(0.5) * w for col, w in score_blocks.items()) / weight_sum
else:
    logger.warning("[5.1] No score inputs available; using neutral blended_score.")
    latest_cross_section["blended_score"] = 0.5

latest_cross_section["rank"] = latest_cross_section["blended_score"].rank(ascending=False, method="first").astype(int)
latest_cross_section = latest_cross_section.sort_values("rank").reset_index(drop=True)

fund_tickers = set(df_fund["ticker"].dropna().unique()) if "df_fund" in globals() and df_fund is not None and not df_fund.empty and "ticker" in df_fund.columns else set()
failed_tickers = EXPERIMENT.get("fundamentals_failed_tickers", [])
matched_tickers = set(df_merged.loc[df_merged["fundamental_matched"], "ticker"].dropna().unique()) if "fundamental_matched" in df_merged.columns else set()
unmatched_tickers = sorted(set(df_market["ticker"].dropna().unique()) - matched_tickers)

summary_rows = [
    {"metric": "market_rows", "value": len(df_market)},
    {"metric": "market_tickers", "value": df_market["ticker"].nunique()},
    {"metric": "date_range", "value": f"{df_market['date'].min().date()} → {df_market['date'].max().date()}"},
    {"metric": "fundamental_rows", "value": len(df_fund) if "df_fund" in globals() and df_fund is not None else 0},
    {"metric": "tickers_with_fundamentals", "value": len(fund_tickers)},
    {"metric": "tickers_matched_in_market", "value": len(matched_tickers)},
    {"metric": "failed_fundamental_tickers", "value": len(failed_tickers)},
    {"metric": "merge_mode", "value": EXPERIMENT.get("merge_mode")},
    {"metric": "row_level_fundamental_coverage", "value": pct_fmt(EXPERIMENT.get("fundamental_merge_coverage", np.nan))},
    {"metric": "target_column", "value": EXPERIMENT.get("target_column")},
]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TABLES_DIR / "Table_V_dataset_summary.csv", index=False)

rank_cols = [c for c in ["rank", "ticker", "date", "adj_close", "blended_score", "valuation_score", "quality_score", "momentum_score", "risk_score", "sws_snowflake_score", "sws_value_score", "sws_future_score", "sws_past_score", "sws_health_score", "sws_income_score", "pe_ratio", "pb_ratio", "ev_ebitda", "roe", "debt_to_equity", "days_since_fundamental"] if c in latest_cross_section.columns]
company_ranking = latest_cross_section[rank_cols].copy()
company_ranking.to_csv(TABLES_DIR / "Table_V_company_ranking.csv", index=False)

snowflake_axis_cols = ["sws_value_score", "sws_future_score", "sws_past_score", "sws_health_score", "sws_income_score"]
snowflake_table = latest_cross_section[["ticker", "rank", "sws_snowflake_score"] + [c for c in snowflake_axis_cols if c in latest_cross_section.columns]].copy()
snowflake_table.to_csv(TABLES_DIR / "Table_V_sws_style_snowflake_scores.csv", index=False)

unmatched_tickers_df = pd.DataFrame({"ticker": unmatched_tickers})
failed_tickers_df = pd.DataFrame({"ticker": failed_tickers})
unmatched_tickers_df.to_csv(TABLES_DIR / "Table_V_unmatched_tickers.csv", index=False)
failed_tickers_df.to_csv(TABLES_DIR / "Table_V_failed_fundamental_tickers.csv", index=False)

print("Dataset summary")
display(summary_df)
print("Top ranked companies")
display(company_ranking.head(10))
print("SWS-style snowflake scores")
display(snowflake_table.head(10))


## 6. 🔎 Exploration

The exploration layer focuses on valuation distributions, score components, data availability, and ticker-level diagnostics. It creates figures that are reused by the final dashboard.

In [ ]:
# ============================================================
# 6.1 - Exploratory charts, SWS-style snowflake, and ticker spotlight
# ============================================================

logger.info("[6.1] Creating exploratory valuation, coverage, and snowflake charts")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig_paths = {}
interactive_figures = {}

selected_company_raw = USER_SELECTION.get("selected_company", "UCG.MI") if "USER_SELECTION" in globals() else "UCG.MI"
selected_company = normalize_ticker(selected_company_raw.replace("_", "."))
if selected_company_raw.lower() == "ucg_mi":
    selected_company = "UCG.MI"
if selected_company not in set(df_model["ticker"].dropna().unique()):
    logger.warning("[6.1] Selected company %s not found; falling back to highest-ranked ticker.", selected_company)
    selected_company = company_ranking.iloc[0]["ticker"]

company_history = df_model.loc[df_model["ticker"].eq(selected_company)].sort_values("date").copy()
latest_snapshot = latest_cross_section.loc[latest_cross_section["ticker"].eq(selected_company)].copy()

valuation_metric = next((c for c in ["upside_to_fair_value", "ev_ebitda", "pe_ratio", "pb_ratio", "blended_score"] if c in latest_cross_section.columns and latest_cross_section[c].notna().sum() >= 3), "blended_score")
quality_metric = next((c for c in ["sws_snowflake_score", "roe", "quality_score", "gross_margin", "net_margin"] if c in latest_cross_section.columns and latest_cross_section[c].notna().sum() >= 3), "blended_score")
size_metric = next((c for c in ["market_cap_est", "adj_close"] if c in latest_cross_section.columns and latest_cross_section[c].notna().sum() >= 3), "adj_close")

if globals().get("PLOTLY_AVAILABLE", False):
    fig = px.histogram(latest_cross_section.dropna(subset=[valuation_metric]), x=valuation_metric, nbins=30, title=f"Distribution of {valuation_metric}", template="plotly_white", color_discrete_sequence=[COLORS["primary"]])
    interactive_figures["valuation_distribution"] = fig
    fig.write_html(FIGURES_DIR / "valuation_distribution.html", include_plotlyjs="cdn")
    fig.show()
else:
    plt.figure(figsize=(9, 5)); sns.histplot(latest_cross_section[valuation_metric].dropna(), bins=30, color=COLORS["primary"]); plt.title(f"Distribution of {valuation_metric}"); plt.tight_layout(); fig_paths["valuation_distribution"] = str(FIGURES_DIR / "valuation_distribution.png"); plt.savefig(fig_paths["valuation_distribution"], bbox_inches="tight"); plt.show()

plot_rank = pd.concat([company_ranking.head(10).assign(group="Top 10"), company_ranking.tail(10).assign(group="Bottom 10")])
if globals().get("PLOTLY_AVAILABLE", False):
    fig = px.bar(plot_rank, x="ticker", y="blended_score", color="group", title="Top and bottom ranked companies by blended score", template="plotly_white", color_discrete_map={"Top 10": COLORS["primary"], "Bottom 10": COLORS["q1"]})
    interactive_figures["top_bottom_ranking"] = fig
    fig.write_html(FIGURES_DIR / "top_bottom_ranking.html", include_plotlyjs="cdn")
    fig.show()
else:
    plt.figure(figsize=(11, 5)); sns.barplot(data=plot_rank, x="ticker", y="blended_score", hue="group", palette={"Top 10": COLORS["primary"], "Bottom 10": COLORS["q1"]}); plt.xticks(rotation=45, ha="right"); plt.title("Top and bottom ranked companies by blended score"); plt.tight_layout(); fig_paths["top_bottom_ranking"] = str(FIGURES_DIR / "top_bottom_ranking.png"); plt.savefig(fig_paths["top_bottom_ranking"], bbox_inches="tight"); plt.show()

if globals().get("PLOTLY_AVAILABLE", False):
    fig = px.scatter(latest_cross_section, x=valuation_metric, y=quality_metric, size=size_metric if size_metric in latest_cross_section.columns else None, hover_name="ticker", color="blended_score", color_continuous_scale="Tealgrn", title=f"Valuation vs quality: {valuation_metric} vs {quality_metric}", template="plotly_white")
    interactive_figures["valuation_quality_scatter"] = fig
    fig.write_html(FIGURES_DIR / "valuation_quality_scatter.html", include_plotlyjs="cdn")
    fig.show()
else:
    plt.figure(figsize=(8, 6)); scatter_size = pd.to_numeric(latest_cross_section[size_metric], errors="coerce").rank(pct=True).fillna(0.5) * 250 + 30; plt.scatter(latest_cross_section[valuation_metric], latest_cross_section[quality_metric], s=scatter_size, c=latest_cross_section["blended_score"], cmap="viridis", alpha=0.75, edgecolor="white"); plt.colorbar(label="blended_score"); plt.xlabel(valuation_metric); plt.ylabel(quality_metric); plt.title(f"Valuation vs quality: {valuation_metric} vs {quality_metric}"); plt.tight_layout(); fig_paths["valuation_quality_scatter"] = str(FIGURES_DIR / "valuation_quality_scatter.png"); plt.savefig(fig_paths["valuation_quality_scatter"], bbox_inches="tight"); plt.show()

coverage_counts = pd.DataFrame({"status": ["matched", "unmatched"], "tickers": [len(matched_tickers), len(unmatched_tickers)]})
if globals().get("PLOTLY_AVAILABLE", False):
    fig = px.bar(coverage_counts, x="status", y="tickers", color="status", title="Ticker-level fundamental coverage", template="plotly_white", color_discrete_map={"matched": COLORS["primary"], "unmatched": COLORS["q1"]})
    interactive_figures["coverage_chart"] = fig
    fig.write_html(FIGURES_DIR / "coverage_chart.html", include_plotlyjs="cdn")
    fig.show()
else:
    plt.figure(figsize=(7, 4)); sns.barplot(data=coverage_counts, x="status", y="tickers", hue="status", palette={"matched": COLORS["primary"], "unmatched": COLORS["q1"]}, legend=False); plt.title("Ticker-level fundamental coverage"); plt.tight_layout(); fig_paths["coverage_chart"] = str(FIGURES_DIR / "coverage_chart.png"); plt.savefig(fig_paths["coverage_chart"], bbox_inches="tight"); plt.show()

snowflake_axis_labels = ["Value", "Future", "Past", "Health", "Income"]
snowflake_cols = ["sws_value_score", "sws_future_score", "sws_past_score", "sws_health_score", "sws_income_score"]
selected_snowflake = latest_snapshot[snowflake_cols].iloc[0].fillna(0).tolist() if not latest_snapshot.empty and all(c in latest_snapshot.columns for c in snowflake_cols) else [0, 0, 0, 0, 0]
if globals().get("PLOTLY_AVAILABLE", False):
    fig = go.Figure(data=go.Scatterpolar(r=selected_snowflake + [selected_snowflake[0]], theta=snowflake_axis_labels + [snowflake_axis_labels[0]], fill="toself", name=selected_company, line_color=COLORS["primary"]))
    fig.update_layout(title=f"SWS-style snowflake profile: {selected_company}", polar=dict(radialaxis=dict(visible=True, range=[0, 6])), template="plotly_white")
    interactive_figures["snowflake_profile"] = fig
    fig.write_html(FIGURES_DIR / "snowflake_profile.html", include_plotlyjs="cdn")
    fig.show()
else:
    angles = np.linspace(0, 2 * np.pi, len(snowflake_axis_labels), endpoint=False).tolist(); values = selected_snowflake + selected_snowflake[:1]; angles += angles[:1]; fig_ax = plt.figure(figsize=(6, 6)); ax = plt.subplot(111, polar=True); ax.plot(angles, values, color=COLORS["primary"], linewidth=2); ax.fill(angles, values, color=COLORS["primary"], alpha=0.25); ax.set_xticks(angles[:-1]); ax.set_xticklabels(snowflake_axis_labels); ax.set_ylim(0, 6); plt.title(f"SWS-style snowflake profile: {selected_company}"); plt.tight_layout(); fig_paths["snowflake_profile"] = str(FIGURES_DIR / "snowflake_profile.png"); plt.savefig(fig_paths["snowflake_profile"], bbox_inches="tight"); plt.show()

spotlight_cols = [c for c in ["rank", "ticker", "date", "adj_close", "blended_score", "sws_snowflake_score", "sws_value_score", "sws_future_score", "sws_past_score", "sws_health_score", "sws_income_score", "valuation_score", "quality_score", "momentum_score", "risk_score", valuation_metric, quality_metric, "effective_fundamental_date", "days_since_fundamental"] if c in latest_snapshot.columns]
latest_snapshot[spotlight_cols].to_csv(TABLES_DIR / "Table_VI_selected_ticker_spotlight.csv", index=False)

print("selected_company:", selected_company)
display(latest_snapshot[spotlight_cols])


## 7. 🧪 Diagnostics

Diagnostics quantify merge quality, data freshness, missingness, and outlier risks so the final interpretation does not hide data-quality constraints.

In [ ]:
# ============================================================
# 7.1 - Merge, freshness, and missingness diagnostics
# ============================================================

logger.info("[7.1] Running diagnostics")

numeric_cols = df_model.select_dtypes(include="number").columns.tolist()
missingness_table = (
    df_model[numeric_cols].isna().mean().sort_values(ascending=False).reset_index()
    .rename(columns={"index": "feature", 0: "pct_missing"})
)
missingness_table.to_csv(TABLES_DIR / "Table_VII_missingness_full.csv", index=False)

if df_merged["fundamental_matched"].any():
    staleness_by_ticker = (
        df_merged.loc[df_merged["fundamental_matched"]]
        .groupby("ticker")["days_since_fundamental"]
        .agg(["count", "median", "mean", "max"])
        .reset_index()
        .sort_values("median", ascending=False)
    )
else:
    staleness_by_ticker = pd.DataFrame(columns=["ticker", "count", "median", "mean", "max"])
staleness_by_ticker.to_csv(TABLES_DIR / "Table_VII_staleness_by_ticker.csv", index=False)

qa_extended = pd.DataFrame([
    {"check": "df_model_non_empty", "status": "PASS" if not df_model.empty else "FAIL", "details": f"rows={len(df_model)}"},
    {"check": "latest_cross_section_non_empty", "status": "PASS" if not latest_cross_section.empty else "FAIL", "details": f"rows={len(latest_cross_section)}"},
    {"check": "no_duplicate_market_keys", "status": "PASS" if not df_market.duplicated(["ticker", "date"]).any() else "FAIL", "details": f"duplicates={int(df_market.duplicated(['ticker','date']).sum())}"},
    {"check": "no_lookahead_fundamentals", "status": "PASS" if not df_merged["days_since_fundamental"].dropna().lt(0).any() else "FAIL", "details": "effective_fundamental_date <= market date"},
    {"check": "score_available", "status": "PASS" if latest_cross_section["blended_score"].notna().any() else "FAIL", "details": f"score_non_null={int(latest_cross_section['blended_score'].notna().sum())}"},
])
qa_extended.to_csv(TABLES_DIR / "Table_VII_extended_qa.csv", index=False)

plt.figure(figsize=(9, 7))
miss_plot = missingness_table.head(20).iloc[::-1]
sns.barplot(data=miss_plot, x="pct_missing", y="feature", color=COLORS["accent"])
plt.title("Top missing numeric fields")
plt.tight_layout()
fig_paths["missingness_top20"] = str(FIGURES_DIR / "missingness_top20.png")
plt.savefig(fig_paths["missingness_top20"], bbox_inches="tight")
plt.show()

print("Extended QA")
display(qa_extended)
print("Highest staleness tickers")
display(staleness_by_ticker.head(15))


## 8. 💰 Valuation / Baseline Models

This section produces an interpretable, non-black-box valuation scorecard and a portfolio-style long/short diagnostic inspired by the repository's strategy-evaluation notebooks.

In [ ]:
# ============================================================
# 8.1 - SWS-inspired fair value estimates, scorecard ranking, and portfolio baseline
# ============================================================

logger.info("[8.1] Building fair-value estimates, valuation ranking, and long/short baseline")

def estimate_intrinsic_value(row, peer_pe=np.nan, peer_pb=np.nan, discount_rate=0.09, terminal_growth=0.025, forecast_years=10):
    price = row.get("adj_close", np.nan)
    shares = row.get("shares_outstanding", np.nan)
    fcf = row.get("free_cash_flow", np.nan)
    revenue_growth = row.get("revenue_growth", np.nan)
    roe = row.get("roe", np.nan)
    bvps = row.get("book_value_per_share", np.nan)
    eps = row.get("eps", np.nan)
    growth = np.nan_to_num(revenue_growth, nan=0.04)
    growth = float(np.clip(growth, -0.05, 0.15))
    model_used = None
    fair_values = []

    # 2-stage FCF model: fade current growth toward terminal growth over 10 years.
    if pd.notna(fcf) and pd.notna(shares) and shares > 0 and fcf > 0 and discount_rate > terminal_growth:
        yearly_growth = np.linspace(growth, terminal_growth, forecast_years)
        projected_fcf = []
        current_fcf = fcf
        for g in yearly_growth:
            current_fcf *= (1 + g)
            projected_fcf.append(current_fcf)
        pv_fcf = sum(cf / ((1 + discount_rate) ** (i + 1)) for i, cf in enumerate(projected_fcf))
        terminal_value = projected_fcf[-1] * (1 + terminal_growth) / (discount_rate - terminal_growth)
        pv_terminal = terminal_value / ((1 + discount_rate) ** forecast_years)
        fair_values.append((pv_fcf + pv_terminal) / shares)
        model_used = "two_stage_fcf"

    # Excess returns model fallback for financial-like firms / weak FCF availability.
    if pd.notna(bvps) and bvps > 0 and pd.notna(roe) and discount_rate > terminal_growth:
        excess_return_value = bvps + ((roe - discount_rate) * bvps) / (discount_rate - terminal_growth)
        if np.isfinite(excess_return_value) and excess_return_value > 0:
            fair_values.append(excess_return_value)
            model_used = model_used or "excess_returns"

    # Relative valuation fallback.
    relative_values = []
    if pd.notna(eps) and eps > 0 and pd.notna(peer_pe) and peer_pe > 0:
        relative_values.append(eps * peer_pe)
    if pd.notna(bvps) and bvps > 0 and pd.notna(peer_pb) and peer_pb > 0:
        relative_values.append(bvps * peer_pb)
    if relative_values:
        fair_values.append(float(np.nanmedian(relative_values)))
        model_used = model_used or "relative_value"

    fair_value = float(np.nanmedian(fair_values)) if fair_values else np.nan
    upside = (fair_value / price - 1) if pd.notna(fair_value) and pd.notna(price) and price > 0 else np.nan
    return pd.Series({"fair_value_estimate": fair_value, "upside_to_fair_value": upside, "fair_value_model": model_used or "insufficient_data"})

peer_pe = numeric_median(latest_cross_section.query("pe_ratio > 0") if "pe_ratio" in latest_cross_section.columns else latest_cross_section, "pe_ratio")
peer_pb = numeric_median(latest_cross_section.query("pb_ratio > 0") if "pb_ratio" in latest_cross_section.columns else latest_cross_section, "pb_ratio")
fair_value_outputs = latest_cross_section.apply(estimate_intrinsic_value, axis=1, peer_pe=peer_pe, peer_pb=peer_pb)
latest_cross_section = pd.concat([latest_cross_section.drop(columns=[c for c in fair_value_outputs.columns if c in latest_cross_section.columns], errors="ignore"), fair_value_outputs], axis=1)
if latest_cross_section["upside_to_fair_value"].notna().any():
    latest_cross_section["fair_value_score"] = safe_rank(latest_cross_section["upside_to_fair_value"], ascending=False)
    latest_cross_section["blended_score"] = 0.85 * latest_cross_section["blended_score"].fillna(0.5) + 0.15 * latest_cross_section["fair_value_score"].fillna(0.5)
    latest_cross_section["rank"] = latest_cross_section["blended_score"].rank(ascending=False, method="first").astype(int)
    latest_cross_section = latest_cross_section.sort_values("rank").reset_index(drop=True)

rank_cols = [c for c in ["rank", "ticker", "date", "adj_close", "fair_value_estimate", "upside_to_fair_value", "fair_value_model", "blended_score", "valuation_score", "quality_score", "momentum_score", "risk_score", "sws_snowflake_score", "pe_ratio", "pb_ratio", "ev_ebitda", "roe", "debt_to_equity", "revenue_growth"] if c in latest_cross_section.columns]
company_ranking = latest_cross_section[rank_cols].copy()
company_ranking.to_csv(TABLES_DIR / "Table_V_company_ranking.csv", index=False)
valuation_output = company_ranking.copy()
valuation_output.to_csv(TABLES_DIR / "Table_VIII_intrinsic_value_estimates.csv", index=False)

scorecard_cols = [c for c in ["rank", "ticker", "date", "adj_close", "fair_value_estimate", "upside_to_fair_value", "fair_value_model", "blended_score", "sws_snowflake_score", "valuation_score", "quality_score", "momentum_score", "risk_score", "pe_ratio", "pb_ratio", "ev_ebitda", "roe", "debt_to_equity", "revenue_growth"] if c in company_ranking.columns]
top_ranked = company_ranking.head(20).copy()
bottom_ranked = company_ranking.tail(20).sort_values("rank", ascending=False).copy()
top_ranked.to_csv(TABLES_DIR / "Table_VIII_top_ranked_companies.csv", index=False)
bottom_ranked.to_csv(TABLES_DIR / "Table_VIII_bottom_ranked_companies.csv", index=False)

historical_score = df_model.copy()
historical_components = []
for src, dst, ascending in [
    ("pe_ratio", "hist_value_pe", True), ("pb_ratio", "hist_value_pb", True), ("ev_ebitda", "hist_value_ev", True),
    ("roe", "hist_quality_roe", False), ("gross_margin", "hist_quality_gm", False),
    ("ret_126d", "hist_momentum", False), ("vol_63d", "hist_risk", True),
]:
    if src in historical_score.columns:
        historical_score[dst] = historical_score.groupby("date")[src].rank(pct=True, ascending=ascending)
        historical_components.append(dst)
if historical_components:
    historical_score["historical_blended_score"] = historical_score[historical_components].mean(axis=1)
else:
    historical_score["historical_blended_score"] = historical_score.groupby("date")["adj_close"].rank(pct=True)

def performance_stats_from_returns(returns, periods_per_year=12):
    returns = pd.Series(returns).dropna()
    if returns.empty:
        return {"annual_return": np.nan, "annual_volatility": np.nan, "sharpe": np.nan, "max_drawdown": np.nan, "hit_rate": np.nan}
    annual_return = (1 + returns.mean()) ** periods_per_year - 1
    annual_vol = returns.std() * np.sqrt(periods_per_year)
    equity = (1 + returns).cumprod()
    drawdown = equity / equity.cummax() - 1
    return {"annual_return": float(annual_return), "annual_volatility": float(annual_vol), "sharpe": float(annual_return / annual_vol) if annual_vol and not np.isclose(annual_vol, 0) else np.nan, "max_drawdown": float(drawdown.min()) if not drawdown.empty else np.nan, "hit_rate": float((returns > 0).mean())}

rebalance_step = max(int(EXPERIMENT.get("target_horizon_days", 21)), 21)
all_dates = pd.Index(sorted(historical_score["date"].dropna().unique()))
rebalance_dates = set(all_dates[::rebalance_step])
backtest_rows = []
for date, group in historical_score[historical_score["date"].isin(rebalance_dates)].groupby("date"):
    group = group.dropna(subset=["historical_blended_score", target_col])
    if len(group) < 5:
        continue
    top_cut = group["historical_blended_score"].quantile(0.8)
    bottom_cut = group["historical_blended_score"].quantile(0.2)
    long_ret = group.loc[group["historical_blended_score"] >= top_cut, target_col].mean()
    short_ret = group.loc[group["historical_blended_score"] <= bottom_cut, target_col].mean()
    ew_ret = group[target_col].mean()
    backtest_rows.append({"date": date, "long_return": long_ret, "short_return": short_ret, "long_short_return": long_ret - short_ret, "equal_weight_return": ew_ret, "n_assets": len(group)})

valuation_backtest = pd.DataFrame(backtest_rows).sort_values("date")
if not valuation_backtest.empty:
    valuation_backtest["long_short_equity"] = (1 + valuation_backtest["long_short_return"].fillna(0)).cumprod()
    valuation_backtest["equal_weight_equity"] = (1 + valuation_backtest["equal_weight_return"].fillna(0)).cumprod()
periods_per_year = max(1, int(252 / rebalance_step))
performance_table = pd.DataFrame([
    {"strategy": "valuation_long_short", **performance_stats_from_returns(valuation_backtest["long_short_return"] if not valuation_backtest.empty else [], periods_per_year)},
    {"strategy": "equal_weight", **performance_stats_from_returns(valuation_backtest["equal_weight_return"] if not valuation_backtest.empty else [], periods_per_year)},
])
valuation_backtest.to_csv(TABLES_DIR / "Table_VIII_valuation_backtest.csv", index=False)
performance_table.to_csv(TABLES_DIR / "Table_VIII_performance_summary.csv", index=False)

plt.figure(figsize=(10, 5))
if not valuation_backtest.empty:
    plt.plot(valuation_backtest["date"], valuation_backtest["long_short_equity"], label="Valuation long/short", color=COLORS["primary"])
    plt.plot(valuation_backtest["date"], valuation_backtest["equal_weight_equity"], label="Equal weight", color=COLORS["accent"])
plt.title("Portfolio-style valuation diagnostic")
plt.ylabel("Growth of 1")
plt.legend()
plt.tight_layout()
fig_paths["valuation_backtest_equity"] = str(FIGURES_DIR / "valuation_backtest_equity.png")
plt.savefig(fig_paths["valuation_backtest_equity"], bbox_inches="tight")
plt.show()

print("Top ranked companies")
display(top_ranked[scorecard_cols].head(10))
print("Bottom ranked companies")
display(bottom_ranked[scorecard_cols].head(10))
print("Performance summary")
display(performance_table)


## 9. 🤖 Advanced Models / Extensions

A lightweight, time-split model is included only as a diagnostic overlay. The valuation scorecard remains the primary interpretable output.

In [ ]:
# ============================================================
# 9.1 - Time-split baseline model diagnostic
# ============================================================

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logger.info("[9.1] Running baseline time-split model")

candidate_features = [
    "blended_score", "historical_blended_score", "ret_21d", "ret_126d", "mom_12_1", "vol_63d",
    "pe_ratio", "pb_ratio", "ev_ebitda", "roe", "roa", "debt_to_equity", "gross_margin", "net_margin", "revenue_growth",
]
# blended_score is latest-only, so use historical_blended_score in the panel.
model_frame = historical_score.copy()
model_features = [c for c in candidate_features if c in model_frame.columns and model_frame[c].notna().sum() > 50]
model_metrics = pd.DataFrame()
model_predictions = pd.DataFrame()
if len(model_features) >= 2 and model_frame[target_col].notna().sum() > 100:
    model_data = model_frame.dropna(subset=model_features + [target_col]).sort_values("date").copy()
    split_date = model_data["date"].quantile(0.75)
    train = model_data[model_data["date"] <= split_date]
    test = model_data[model_data["date"] > split_date]
    if len(train) >= 50 and len(test) >= 20:
        ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
        ridge_model.fit(train[model_features], train[target_col])
        model_predictions = test[["date", "ticker", target_col]].copy()
        model_predictions["prediction"] = ridge_model.predict(test[model_features])
        model_metrics = pd.DataFrame([{
            "model": "ridge_time_split",
            "n_train": len(train),
            "n_test": len(test),
            "split_date": split_date,
            "rmse": float(np.sqrt(mean_squared_error(test[target_col], model_predictions["prediction"]))),
            "mae": float(mean_absolute_error(test[target_col], model_predictions["prediction"])),
            "r2": float(r2_score(test[target_col], model_predictions["prediction"])),
        }])
    else:
        model_metrics = pd.DataFrame([{"model": "ridge_time_split", "status": "insufficient post-split rows"}])
else:
    model_metrics = pd.DataFrame([{"model": "ridge_time_split", "status": "insufficient features or target observations"}])

model_metrics.to_csv(TABLES_DIR / "Table_IX_model_metrics.csv", index=False)
model_predictions.to_csv(TABLES_DIR / "Table_IX_model_predictions.csv", index=False)
display(model_metrics)


## 10. 🧩 Ablation / Sensitivity

Ablation shows which score blocks drive rankings and whether the conclusion is robust to dropping valuation, quality, momentum, or risk components.

In [ ]:
# ============================================================
# 10.1 - Score block ablation
# ============================================================

logger.info("[10.1] Running score ablation")

available_score_blocks = [c for c in ["valuation_score", "quality_score", "momentum_score", "risk_score"] if c in latest_cross_section.columns]
ablation_rows = []
if available_score_blocks:
    baseline_rank = latest_cross_section.set_index("ticker")["blended_score"].rank(ascending=False, method="first")
    for dropped in [None] + available_score_blocks:
        use_cols = [c for c in available_score_blocks if c != dropped]
        if not use_cols:
            continue
        score = latest_cross_section[use_cols].mean(axis=1)
        rank = score.rank(ascending=False, method="first")
        rank_series = pd.Series(rank.values, index=latest_cross_section["ticker"])
        corr = baseline_rank.corr(rank_series, method="spearman") if baseline_rank.nunique() > 1 and rank_series.nunique() > 1 else np.nan
        top_names = latest_cross_section.assign(ablation_score=score).sort_values("ablation_score", ascending=False)["ticker"].head(10).tolist()
        ablation_rows.append({"dropped_block": dropped or "none", "n_blocks_used": len(use_cols), "spearman_vs_baseline_rank": corr, "top10": ", ".join(top_names)})
score_ablation = pd.DataFrame(ablation_rows)
score_ablation.to_csv(TABLES_DIR / "Table_X_score_ablation.csv", index=False)
display(score_ablation)


## 11. ✅ Validation / Scenario Analysis

Scenario analysis converts the scorecard into conservative/base/bull expected-return views for the selected ticker and the ranked universe.

In [ ]:
# ============================================================
# 11.1 - Scenario analysis
# ============================================================

logger.info("[11.1] Building scenario analysis")

scenario_base = latest_cross_section[["ticker", "blended_score"]].copy()
scenario_base["score_centered"] = scenario_base["blended_score"] - scenario_base["blended_score"].median()
scenario_base["bear_expected_return"] = -0.05 + 0.10 * scenario_base["score_centered"]
scenario_base["base_expected_return"] = 0.03 + 0.18 * scenario_base["score_centered"]
scenario_base["bull_expected_return"] = 0.10 + 0.25 * scenario_base["score_centered"]
scenario_base = scenario_base.sort_values("base_expected_return", ascending=False).reset_index(drop=True)
scenario_base.to_csv(TABLES_DIR / "Table_XI_scenario_expected_returns.csv", index=False)

selected_scenario = scenario_base[scenario_base["ticker"].eq(selected_company)].copy()
if selected_scenario.empty:
    selected_scenario = scenario_base.head(1).copy()

scenario_top = scenario_base.head(20).copy()
x = np.arange(len(scenario_top))
width = 0.25
plt.figure(figsize=(12, 5))
plt.bar(x - width, scenario_top["bear_expected_return"], width, label="Bear", color=COLORS["q1"])
plt.bar(x, scenario_top["base_expected_return"], width, label="Base", color=COLORS["primary"])
plt.bar(x + width, scenario_top["bull_expected_return"], width, label="Bull", color=COLORS["accent"])
plt.xticks(x, scenario_top["ticker"], rotation=45, ha="right")
plt.ylabel("Expected return")
plt.title("Top-ranked scenario expected returns")
plt.legend()
plt.tight_layout()
fig_paths["scenario_expected_returns"] = str(FIGURES_DIR / "scenario_expected_returns.png")
plt.savefig(fig_paths["scenario_expected_returns"], bbox_inches="tight")
plt.show()

print("Selected ticker scenario")
display(selected_scenario)


## 12. 🔬 Interpretability / Drivers

Driver analysis explains the ranking in terms of score components and observable financial metrics.

In [ ]:
# ============================================================
# 12.1 - Ranking drivers
# ============================================================

logger.info("[12.1] Computing ranking drivers")

driver_candidates = [c for c in ["valuation_score", "quality_score", "momentum_score", "risk_score", "pe_ratio", "pb_ratio", "ev_ebitda", "roe", "roa", "debt_to_equity", "revenue_growth", "vol_63d", "ret_126d"] if c in latest_cross_section.columns]
driver_rows = []
for col in driver_candidates:
    valid = latest_cross_section[[col, "blended_score"]].dropna()
    if len(valid) >= 3:
        corr = valid[col].corr(valid["blended_score"], method="spearman") if valid[col].nunique() > 1 and valid["blended_score"].nunique() > 1 else np.nan
        driver_rows.append({"driver": col, "corr_with_blended_score": corr, "coverage": len(valid) / len(latest_cross_section)})
driver_table = pd.DataFrame(driver_rows).sort_values("corr_with_blended_score", key=lambda s: s.abs(), ascending=False) if driver_rows else pd.DataFrame(columns=["driver", "corr_with_blended_score", "coverage"])
driver_table.to_csv(TABLES_DIR / "Table_XII_ranking_drivers.csv", index=False)

plt.figure(figsize=(9, 6))
plot_drivers = driver_table.head(15).iloc[::-1]
colors = [COLORS["primary"] if v >= 0 else COLORS["q1"] for v in plot_drivers["corr_with_blended_score"].fillna(0)]
plt.barh(plot_drivers["driver"], plot_drivers["corr_with_blended_score"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Main ranking drivers (Spearman correlation)")
plt.tight_layout()
fig_paths["ranking_drivers"] = str(FIGURES_DIR / "ranking_drivers.png")
plt.savefig(fig_paths["ranking_drivers"], bbox_inches="tight")
plt.show()

display(driver_table.head(15))


## 13. 🛡️ Robustness Checks

The robustness layer summarizes residual data and methodology risks that should be reviewed before investment use.

In [ ]:
# ============================================================
# 13.1 - Robustness checks
# ============================================================

logger.info("[13.1] Running robustness checks")

robustness_checks = []
robustness_checks.append({"check": "fundamental_row_coverage", "status": "PASS" if EXPERIMENT.get("fundamental_merge_coverage", 0) >= FUNDAMENTALS_CONFIG["minimum_match_coverage_warning"] or EXPERIMENT.get("merge_mode") == "market_only" else "WARN", "value": EXPERIMENT.get("fundamental_merge_coverage", 0)})
robustness_checks.append({"check": "failed_ticker_count", "status": "PASS" if len(failed_tickers) == 0 else "WARN", "value": len(failed_tickers)})
robustness_checks.append({"check": "unmatched_ticker_count", "status": "PASS" if len(unmatched_tickers) == 0 or EXPERIMENT.get("merge_mode") == "market_only" else "WARN", "value": len(unmatched_tickers)})
robustness_checks.append({"check": "ranking_metric_available", "status": "PASS" if latest_cross_section["blended_score"].notna().any() else "FAIL", "value": int(latest_cross_section["blended_score"].notna().sum())})
robustness_checks.append({"check": "target_available", "status": "PASS" if df_model[target_col].notna().sum() > 0 else "WARN", "value": int(df_model[target_col].notna().sum())})
if df_merged["fundamental_matched"].any():
    robustness_checks.append({"check": "median_fundamental_staleness_days", "status": "PASS" if df_merged["days_since_fundamental"].median(skipna=True) <= MERGE_CONFIG["staleness_warning_days"] else "WARN", "value": float(df_merged["days_since_fundamental"].median(skipna=True))})
else:
    robustness_checks.append({"check": "median_fundamental_staleness_days", "status": "WARN", "value": np.nan})

robustness_table = pd.DataFrame(robustness_checks)
robustness_table.to_csv(TABLES_DIR / "Table_XIII_robustness_checks.csv", index=False)
display(robustness_table)


## 14. 📌 Final Dashboard / Conclusion

The final dashboard consolidates data coverage, valuation ranking, diagnostics, selected-ticker spotlight, scenario analysis, portfolio-style validation, and the analyst summary into a shareable HTML artifact.

In [ ]:
# ============================================================
# 14.1 - Final results dashboard and analyst summary
# ============================================================

import base64

logger.info("[14.1] Building final dashboard")

DASHBOARD_DIR = OUTPUT_ROOT / "dashboard"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

def card(label, value, subtitle=""):
    return f"""
    <div class='kpi-card'>
      <div class='kpi-label'>{label}</div>
      <div class='kpi-value'>{value}</div>
      <div class='kpi-subtitle'>{subtitle}</div>
    </div>
    """


def img_tag(path, title):
    path = Path(path)
    if not path.exists():
        return f"<p><strong>{title}</strong>: chart not available.</p>"
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"<h3>{title}</h3><img src='data:image/png;base64,{encoded}' style='width:100%; max-width:1100px; border-radius:10px; margin-bottom:16px;'/>"

median_staleness = df_merged["days_since_fundamental"].median(skipna=True) if "days_since_fundamental" in df_merged.columns else np.nan
fund_ticker_count = len(fund_tickers)
row_coverage = EXPERIMENT.get("fundamental_merge_coverage", 0.0)
market_start = df_market["date"].min().date()
market_end = df_market["date"].max().date()

kpi_html = "".join([
    card("Tickers analyzed", f"{df_market['ticker'].nunique():,}", f"{market_start} → {market_end}"),
    card("With fundamentals", f"{fund_ticker_count:,}", f"matched tickers: {len(matched_tickers):,}"),
    card("Row merge coverage", f"{row_coverage:.1%}", EXPERIMENT.get("merge_mode", "unknown")),
    card("Median staleness", "n/a" if pd.isna(median_staleness) else f"{median_staleness:.0f} days", "matched rows only"),
    card("Failed tickers", f"{len(failed_tickers):,}", "FMP/cache failures"),
    card("SWS-style score", "n/a" if latest_cross_section["sws_snowflake_score"].isna().all() else f"{latest_cross_section['sws_snowflake_score'].median():.1f}/6", "median snowflake"),
    card("Target", EXPERIMENT.get("target_column", "n/a"), f"horizon={EXPERIMENT.get('target_horizon_days', 'n/a')}d"),
])

if globals().get("interactive_figures"):
    chart_html = "".join(
        fig.to_html(full_html=False, include_plotlyjs=("cdn" if i == 0 else False))
        for i, fig in enumerate(interactive_figures.values())
    )
    chart_html += "".join([
        img_tag(fig_paths.get("missingness_top20", ""), "Missingness diagnostics"),
        img_tag(fig_paths.get("valuation_backtest_equity", ""), "Portfolio-style validation"),
        img_tag(fig_paths.get("scenario_expected_returns", ""), "Scenario expected returns"),
        img_tag(fig_paths.get("ranking_drivers", ""), "Ranking drivers"),
    ])
else:
    chart_html = "".join([
        img_tag(fig_paths.get("valuation_distribution", ""), "Valuation distribution"),
        img_tag(fig_paths.get("top_bottom_ranking", ""), "Top and bottom ranking"),
        img_tag(fig_paths.get("coverage_chart", ""), "Fundamental coverage"),
        img_tag(fig_paths.get("valuation_quality_scatter", ""), "Valuation vs quality"),
        img_tag(fig_paths.get("snowflake_profile", ""), "SWS-style snowflake profile"),
        img_tag(fig_paths.get("missingness_top20", ""), "Missingness diagnostics"),
        img_tag(fig_paths.get("valuation_backtest_equity", ""), "Portfolio-style validation"),
        img_tag(fig_paths.get("scenario_expected_returns", ""), "Scenario expected returns"),
        img_tag(fig_paths.get("ranking_drivers", ""), "Ranking drivers"),
    ])

analyst_summary_lines = []
analyst_summary_lines.append(f"The notebook analyzed {df_market['ticker'].nunique():,} tickers from {market_start} to {market_end}.")
if EXPERIMENT.get("merge_mode") == "market_plus_fundamentals":
    analyst_summary_lines.append(f"Fundamental data were matched to {row_coverage:.1%} of market rows using an availability-date as-of merge with no detected look-ahead violations.")
else:
    analyst_summary_lines.append("Fundamental data were unavailable or insufficient, so the final flow used the market-only fallback while preserving diagnostics for future FMP/cache runs.")
if not company_ranking.empty:
    analyst_summary_lines.append(f"The highest-ranked ticker is {company_ranking.iloc[0]['ticker']} with a blended score of {company_ranking.iloc[0]['blended_score']:.2f}.")
if not performance_table.empty and "sharpe" in performance_table.columns:
    best_perf = performance_table.sort_values("sharpe", ascending=False, na_position="last").head(1)
    if not best_perf.empty and pd.notna(best_perf.iloc[0].get("sharpe", np.nan)):
        analyst_summary_lines.append(f"The portfolio-style diagnostic favors {best_perf.iloc[0]['strategy']} on Sharpe within the available sample.")
analyst_summary_lines.append("Residual risks are concentrated in fundamentals coverage, international ticker mapping, reporting-date availability, and the limited causal interpretation of cross-sectional ranks.")
analyst_summary = " ".join(analyst_summary_lines)

style = f"""
<style>
body {{font-family: Inter, Arial, sans-serif; color: #1f2933;}}
.dashboard {{background: #f7f6f2; padding: 22px; border-radius: 18px;}}
.header {{background: linear-gradient(120deg, {COLORS['primary']}, #003b46); color: white; padding: 24px; border-radius: 16px; margin-bottom: 18px;}}
.header h1 {{margin: 0; font-size: 30px;}}
.header p {{margin: 8px 0 0 0; opacity: .9;}}
.kpi-grid {{display: grid; grid-template-columns: repeat(3, minmax(180px, 1fr)); gap: 14px; margin: 18px 0;}}
.kpi-card {{background: white; padding: 16px; border-radius: 14px; box-shadow: 0 2px 10px rgba(0,0,0,.07); border-left: 5px solid {COLORS['accent']};}}
.kpi-label {{font-size: 12px; text-transform: uppercase; color: #667085; letter-spacing: .05em;}}
.kpi-value {{font-size: 26px; font-weight: 800; margin-top: 4px; color: {COLORS['primary']};}}
.kpi-subtitle {{font-size: 12px; color: #667085; margin-top: 3px;}}
.section {{background: white; padding: 18px; border-radius: 14px; margin: 18px 0; box-shadow: 0 2px 10px rgba(0,0,0,.05);}}
.section h2 {{margin-top: 0; color: {COLORS['primary']};}}
table {{border-collapse: collapse; width: 100%; font-size: 13px;}}
th {{background: {COLORS['primary']}; color: white; padding: 8px;}}
td {{padding: 7px; border-bottom: 1px solid #eee;}}
</style>
"""

snowflake_table_html = snowflake_table.head(15).to_html(index=False, classes="table", float_format=lambda x: f"{x:,.3f}")
valuation_table_html = valuation_output.head(15).to_html(index=False, classes="table", float_format=lambda x: f"{x:,.3f}") if "valuation_output" in globals() else "<p>Intrinsic value estimates unavailable.</p>"
top_table_html = top_ranked[scorecard_cols].head(15).to_html(index=False, classes="table", float_format=lambda x: f"{x:,.3f}")
bottom_table_html = bottom_ranked[scorecard_cols].head(15).to_html(index=False, classes="table", float_format=lambda x: f"{x:,.3f}")
failed_table_html = failed_tickers_df.head(50).to_html(index=False, classes="table") if not failed_tickers_df.empty else "<p>No failed tickers recorded in this run.</p>"
unmatched_table_html = unmatched_tickers_df.head(50).to_html(index=False, classes="table") if not unmatched_tickers_df.empty else "<p>No unmatched tickers recorded in this run.</p>"
spotlight_html = latest_snapshot[spotlight_cols].to_html(index=False, classes="table", float_format=lambda x: f"{x:,.3f}") if not latest_snapshot.empty else "<p>No selected ticker snapshot available.</p>"
robustness_html = robustness_table.to_html(index=False, classes="table", float_format=lambda x: f"{x:,.3f}")

final_dashboard_html = f"""
{style}
<div class='dashboard'>
  <div class='header'>
    <h1>Company Valuation & Market/Fundamentals Dashboard</h1>
    <p>Research-grade integrated valuation scorecard, diagnostics, scenario analysis, and portfolio-style validation.</p>
  </div>
  <div class='kpi-grid'>{kpi_html}</div>
  <div class='section'><h2>Analyst Summary</h2><p>{analyst_summary}</p></div>
  <div class='section'><h2>Selected Ticker Spotlight: {selected_company}</h2>{spotlight_html}</div>
  <div class='section'><h2>SWS-Style Snowflake Scores</h2>{snowflake_table_html}</div>
  <div class='section'><h2>Intrinsic / Relative Fair Value Estimates</h2>{valuation_table_html}</div>
  <div class='section'><h2>Top Ranked Companies</h2>{top_table_html}</div>
  <div class='section'><h2>Bottom Ranked Companies</h2>{bottom_table_html}</div>
  <div class='section'><h2>Coverage Exceptions</h2><h3>Failed FMP/cache tickers</h3>{failed_table_html}<h3>Unmatched tickers</h3>{unmatched_table_html}</div>
  <div class='section'><h2>Robustness Checks</h2>{robustness_html}</div>
  <div class='section'><h2>Charts</h2>{chart_html}</div>
</div>
"""

dashboard_path = DASHBOARD_DIR / "company_valuation_dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as fh:
    fh.write(final_dashboard_html)

final_report = {
    "implemented_changes": [
        "Hardened FMP secret loading and source tracking.",
        "Built df_fund from local cache or FMP income/balance/cash-flow statements with canonical fields and failure logs.",
        "Merged market and fundamentals through conservative effective availability dates with ticker-isolated as-of joins.",
        "Added SWS-inspired snowflake scoring, intrinsic/relative fair-value estimates, analyst scorecard ranking, portfolio-style baseline, diagnostics, ablation, scenario analysis, interpretability, robustness checks, and a Plotly-capable final dashboard.",
    ],
    "critical_issues_found": [
        "Earlier notebook flow could end after assembly without a complete analytical conclusion or investor-style dashboard.",
        "Fundamental availability and accepted-date fields are vendor-dependent; conservative fallback logic remains necessary.",
        "International ticker suffixes remain a live data-quality risk and are surfaced in failed/unmatched ticker tables.",
    ],
    "final_notebook_flow": {
        "secrets": {"available": FMP_API_KEY_AVAILABLE, "source": FMP_SECRET_SOURCE},
        "fundamentals": {"available": EXPERIMENT.get("fundamentals_available"), "source": EXPERIMENT.get("fundamentals_source"), "rows": int(len(df_fund)) if "df_fund" in globals() and df_fund is not None else 0},
        "merge": {"mode": EXPERIMENT.get("merge_mode"), "coverage": EXPERIMENT.get("fundamental_merge_coverage")},
        "results": {"ranked_companies": int(len(company_ranking)), "selected_company": selected_company, "dashboard_path": str(dashboard_path)},
    },
    "analyst_summary": analyst_summary,
    "artifacts": {
        "dashboard_html": str(dashboard_path),
        "company_ranking": str(TABLES_DIR / "Table_V_company_ranking.csv"),
        "intrinsic_value_estimates": str(TABLES_DIR / "Table_VIII_intrinsic_value_estimates.csv"),
        "performance_summary": str(TABLES_DIR / "Table_VIII_performance_summary.csv"),
        "scenario_table": str(TABLES_DIR / "Table_XI_scenario_expected_returns.csv"),
        "robustness_table": str(TABLES_DIR / "Table_XIII_robustness_checks.csv"),
    },
}
report_path = TABLES_DIR / "final_notebook_report.json"
with open(report_path, "w", encoding="utf-8") as fh:
    json.dump(final_report, fh, indent=2, default=str)

print("Dashboard saved to:", dashboard_path)
print(json.dumps(final_report, indent=2, default=str))
display(HTML(final_dashboard_html))
